In [70]:
import os
import sys
import anndata as ad
import numpy as np
import pandas as pd
import scanpy as sc
import seaborn as sns
import scanpy.external as sce
import matplotlib.pyplot as plt
import re
import gseapy as gp
import anndata as ad
import statistics
import tempfile
import sklearn
import cosg
import leidenalg
import muon as mu
import mudata as md
from tqdm import tqdm
from sklearn_ann.kneighbors.annoy import AnnoyTransformer
sc._settings.n_jobs= 24
sc.settings.verbosity = 1
# Adjust Scanpy figure defaults
pd.set_option('display.max_rows', 100)
pd.set_option('display.max_columns', 100)
import warnings
warnings.filterwarnings('ignore')

sc.settings.set_figure_params(dpi=100, fontsize=10, dpi_save=300,
    facecolor = 'white', figsize=(8,8), format='pdf')
import matplotlib as mpl
mpl.rcParams['pdf.fonttype'] = 42
mpl.rcParams['font.family'] = 'Arial'
sns.despine()
mpl.rcParams['lines.linewidth'] = 0.5

<Figure size 800x800 with 0 Axes>

In [10]:
def check_dict_duplicates(dict):
    seen = set()
    dup = any(item in seen or seen.add(item) for lst in dict.values() for item in lst)
    if dup==False:
        return '无重复'
    else:
        return '有重复'

In [13]:
obj_path = '/home/liyanguo/MyImmuCell/05_Ref_Atlas_subpopulation/Level2_Refine_R1/'

In [14]:
marker_dict={
    "Immune cell": ["PTPRC"],
    
    'HSPC':['CD34','SPINK2','KIT',"MECOM", "PROM1","DNTT",'IKZF1','CD7',"SOX4",],
    
    "Lymph prog": ["EBF1","SSBP2","BACH2","PAX5","PRKCE",],
    
    "T": ["CD3D","CD3E","CD3G"],
    "T naive": ["LEF1","IL7R","CCR7", "TCF7","SELL"],
    "CD4+ T": ["CD4",],
    "CD8+ T": ["CD8A", "CD8B", "GZMK", "GZMA", "CCL5",'PRF1','EOMES'],
    "NK": ["NCAM1","FCGR3A",'CD244','KLRD1',
           "GNLY", "NKG7", "CD247",],
    "Adaptive NK":['CD226',"B3GAT1",'KIR3DL1'],
    'MAIT':['KLRB1','TRAV1-2',"MR1"],
    'γδ T':['TRGV9','TRDV2',"TRGV5","TRDV1",'TRGC1','TRDC','KLRC2','KLRF1'],
    'pILC':["ID2",'NFIL3',],
    'ILC1':['NCR1','IL2RB'],#lin-
    'ILC2':["IL7R",'PTGDR2',"GATA3",'ZBTB16','RORA','IL32','MAF',"KLRG1",],#lin-
    'ILC3':['AHR',"KIT",'RUNX2','RORC','NCR2','JUN'],#mature ILC#lin-
    "ILC": ['LTB','RGS1','TNFSF10',"PLCG2", "SYNE1",'RUNX1','TOX','FLT3'],#lin-
    'NKT':['TRAV24'], 
    'Treg':['FOXP3','CTLA4','IL2RA'],
    'Unsigned T':['ANK3','PRKCA','BCL6'],
    'T Develop':['SPI1','ZBTB7B','RUNX3'],
    "T activation": ["CD69", "CD38"],
    
    'B naive': ['MS4A1','CD79A',"TCL1A",],
    'Transitional B': ["MME","CD24", "MSI2","ACSM3",],
    'Atypical memory B': ["TBX21","ITGAX","FCRL5"],
    'Plasma':['JCHAIN',"MZB1","CD38",],
    'Plasmablast': ["XBP1", "PRDM1", "PAX5"],
    
    'DC':['CD1C','HLA-DQA1','FCER1A','CST3','CLEC10A','CLEC9A',"CADM1"],
    'pDC':['CLEC4C','IL3RA'],
    'Monocytes' :['CD14','LYZ','VCAN','FCN1','CSF1R',"FCGR3A",'TCF7L2','LYN'],
    
    'Neutrophile CEACAM8-':['FCGR3B','CSF3R','G0S2','MNDA',"MME",],
    'Neutrophile CEACAM8+':['CEACAM8',"CD24",],
    'Basophil':['GATA2', 'FCER1A', 'IL3RA','ENPP3','MS4A2'],
    'Eosinophil':['RNASE2','MBP','EPX', 'SIGLEC8', 'IL5RA','CD38','CCR3',],
    'Mast cell':['FCER1A','MS4A2','KIT'],
    "G/M prog": ["MPO", "BCL2", "KCNQ5"],
    
    'Platelet':['PF4','PPBP','GP9','TFRC'],
    
    'Proliferative signal':['MKI67','TOP2A','STMN1'],
    'RBC':['HBB','HBA2'],
    'Other':["NKAIN2","NRIP1",]#不太管用的
}
#Protein :       CD114,  CD116,  CD123, CD124, CD125,  CD126, CD203c,  KLRB1,  CD25,   CD29,   CD73,  CD127,  CD119,  CRTH2,  PLZF, CD62L, TdT, CD57,     ThpOK,   PU.1, CRTH2
#Gene Symbol :	CSF3R,   CSF2RA, IL3RA, IL4R,  IL5RA,  IL6R, ENPP3,   CD161,  IL2RA,  ITGB1,  NT5E,  IL7R,  IFNGR1,  PTGDR2, ZBTB16, SELL, DNTT, B3GAT1, ZBTB7B,   SPI1, PTGDR2

#Protein :    CD42a, CD42b, 
#Gene Symbol :GP9, GP1BA

#TCR : Vα7.2
#Gene : TRAV1-2

#GCSF=CSF3, for neutrophils
#GM-CSF=CSF2, granulocyte-macrophage colony-stimulating factor
#M-CSF=CSF1, macrophage colony-stimulating factor


In [15]:
adt_dict={
    "CD45RA": ["CD45RA"],
    
    'B': ['IgD','IgM',"CD19.1",'CD27.1','CD137'],
    
    "ILC|NK|NKT|γδ T|": ["CD56","CD161",'CD16',],
    
    "T": ["CD3","CD4.1","CD8",'CD27.1','CD28.1','CD134','CD272'],
    "Naive/Memory T": ["CD45RA","CD127","CD62L",'CD197',],
    "Treg":['CD25','CD137','CD357',],
    "Th1|ILC":["CD183",'CD186','CD366'],
    "Th17|ILC":['CD161','CD196',],
    "Tfh|ILC|Effector":['CD185','CD278','CD279',],

    'DC':['CD11c','HLA-DR',"CD183",'CD197'],

    'Monocytes' :['CD14.1','CD16'],
}

# 1. Basophil Cell Refine

In [254]:
celltype="Basophil"

In [255]:
adata = sc.read_h5ad(f"{obj_path}{celltype}/{celltype}_after_cluster_scVI.h5ad")
R_data_path = f"{obj_path}{celltype}"
sc.settings.figdir=f"{obj_path}{celltype}"

In [256]:
sc.pp.normalize_total(adata, target_sum=1e4)
sc.pp.log1p(adata)

In [257]:
del adata.obs['Celltype_L1_L2_Refine']

In [ ]:
sc.pl.umap(adata, color=['receptor_type','receptor_type_BCR'],
           frameon=False,
           legend_fontsize=4, legend_fontoutline=2, 
           size=4,
           legend_loc='on data')

In [260]:
pd.DataFrame(adata.obs[['L2_leiden_scVI_0.5', 'L2_leiden_scVI_1', 'L2_leiden_scVI_1.5', 'L2_leiden_scVI_2']]).to_csv(f'{R_data_path}/Level2_scVI_ForClustree.csv')
os.system(f'/home/liyanguo/anaconda3/envs/R/bin/Rscript ref_clustree.R {R_data_path} &')

0

In [274]:
groupby = "L2_leiden_scVI_0.5"

In [275]:
xlsx = pd.ExcelWriter(f"{obj_path}{celltype}/{groupby}_freq_table.xlsx")
for group in ['SampleID', 'DonorID','scDblFinder.class','Phase',
              'Immune_All_High', 'Immune_All_Low', 'Adult_Human_Blood', 'Adult_Human_Bone_marrow',
              'AIFI_L1', 'AIFI_L2', 'AIFI_L3', 'predicted.celltype.l1', 'predicted.celltype.l2',
              'predicted.celltype.l3',
              'receptor_subtype', #TCR
              'VJ_1_v_call', 'VJ_1_j_call', 
              'VDJ_1_v_call', 'VDJ_1_j_call',
              'receptor_type_BCR', #BCR
              'VJ_1_v_call_BCR','VJ_1_j_call_BCR', 
              'VDJ_1_v_call_BCR', 'VDJ_1_j_call_BCR']:
    freq_table_multi = adata.obs.groupby([groupby, group]).size()
    pd.DataFrame(freq_table_multi).to_excel(xlsx,sheet_name=group)
xlsx.close()

In [ ]:
#数据簇间信息比较
fig, axs = plt.subplots(5, 2, figsize=(18, 12),constrained_layout=True)
plt.subplots_adjust(hspace=1,wspace=1)
sc.pl.violin(adata, keys='nCount_RNA', groupby=groupby, rotation=90,size=0.1,palette="pastel",linewidth=0,ax=axs[0,0], show=False)
sc.pl.violin(adata, keys='nFeature_RNA', groupby=groupby, rotation=90,size=0.1,palette="pastel",linewidth=0,ax=axs[0,1], show=False)
sc.pl.violin(adata, keys='log10GenesPerUMI', groupby=groupby, rotation=90,size=0.1,palette="pastel",linewidth=0,ax=axs[1,0], show=False)
sc.pl.violin(adata, keys='pct_counts_in_top_20_genes', groupby=groupby, rotation=90,size=0.1,palette="pastel",linewidth=0,ax=axs[1,1], show=False)

sc.pl.violin(adata, keys='percent_apop', groupby=groupby, rotation=90,size=0.1,palette="pastel",linewidth=0,ax=axs[2,0], show=False)
sc.pl.violin(adata, keys='percent_ribo', groupby=groupby, rotation=90,size=0.1,palette="pastel",linewidth=0,ax=axs[2,1], show=False)
sc.pl.violin(adata, keys='percent_ieg', groupby=groupby, rotation=90,size=0.1,palette="pastel",linewidth=0,ax=axs[3,0], show=False)
sc.pl.violin(adata, keys='percent_oxphos', groupby=groupby, rotation=90,size=0.1,palette="pastel",linewidth=0,ax=axs[3,1], show=False)
sc.pl.violin(adata, keys='percent_hemo', groupby=groupby, rotation=90,size=0.1,palette="pastel",linewidth=0,ax=axs[4,0], show=False)
sc.pl.violin(adata, keys='G2M.Score', groupby=groupby, rotation=90,size=0.1,palette="pastel",linewidth=0,ax=axs[4,1], show=False)

fig.tight_layout()
plt.savefig(f'{obj_path}{celltype}/{groupby}_scVI_L2_metadata.png')

In [277]:
sc.tl.dendrogram(adata,groupby=groupby,use_rep='X_scVI')

In [ ]:
#cosg差异
cosg.cosg(adata, key_added=f'cosg_{groupby}', groupby=groupby,
          mu=10,n_genes_user=100,remove_lowly_expressed=True,
         )
df_tmp = pd.DataFrame(adata.uns[f'cosg_{groupby}']['names'])
df_tmp.to_csv(f"{obj_path}{celltype}/cosg_{groupby}.csv")
#cosg差异作图
df_tmp=pd.DataFrame(adata.uns[f'cosg_{groupby}']['names'][:8,]).T
df_tmp=df_tmp.reindex(adata.uns['dendrogram_'+groupby]['categories_ordered'])
marker_genes_list={idx: list(row.values) for idx, row in df_tmp.iterrows()}
marker_genes_list = {k: v for k, v in marker_genes_list.items() if not any(isinstance(x, float) for x in v)}
sc.pl.dotplot(adata, marker_genes_list,
             groupby=groupby,
             dendrogram=True,
             swap_axes=False,
             standard_scale='var',
             save=f'cosg_{groupby}',
             cmap='Spectral_r')

#注意此处的标准化standard_scale有多种可选，var / group

In [279]:
#scanpy 差异结果
sc.tl.rank_genes_groups(adata, groupby, method="wilcoxon",pts=True,key_added=f'rank_genes_{groupby}')
result_df = sc.get.rank_genes_groups_df(adata,group=None, pval_cutoff=1e-3, log2fc_min=0.5,key=f'rank_genes_{groupby}')
result_df.to_csv(f"{obj_path}{celltype}/wilcoxon_{celltype}_{groupby}.csv")

Calculating cluster 14
Calculating cluster 15


In [ ]:
sc.pl.rank_genes_groups_dotplot(adata,standard_scale='var', dendrogram=True,swap_axes=False,cmap='Spectral_r',n_genes=10,key=f'rank_genes_{groupby}')

In [281]:
os.system(f'nohup /home/liyanguo/anaconda3/envs/R/bin/Rscript DEG_ROC.R {R_data_path} {celltype} {groupby} &')

0

In [282]:
FeatureMatrix = pd.DataFrame(adata.uns[f'cosg_{groupby}']['names'])

In [ ]:
%%R -i FeatureMatrix,R_data_path,groupby
source('GetEnrich.R')
GetEnrich(FeatureMatrix,plot_term_number = 10,R_data_path=R_data_path,groupby=groupby,
          gmtfile="/localdisk/immune/Marker/msigdb_v2024.1.Hs_GMTs/h.all.v2024.1.Hs.symbols.gmt")
#"/localdisk/immune/Marker/Immunity_PBMC.gmt"
#"/localdisk/immune/Marker/sepsis_PBMC.gmt"
#"/localdisk/immune/Marker/msigdb_v2024.1.Hs_GMTs/c5.go.bp.v2024.1.Hs.symbols.gmt"
#"/localdisk/immune/Marker/msigdb_v2024.1.Hs_GMTs/c2.cp.kegg_legacy.v2024.1.Hs.symbols.gmt"

In [ ]:
sc.tl.score_genes(adata,['ENPP3','CD63','IL3RA','CCR3','LAMP1'])
fig, axs = plt.subplots(ncols=2, nrows=1, figsize=(10, 4))
sc.pl.umap(adata, color='score',legend_fontsize=4, legend_fontoutline=2,size=4,legend_loc='on data',show=False, ax=axs[0])
sc.pl.violin(adata,'score',groupby=groupby,size=0.5,show=False, ax=axs[1])
plt.show()

In [ ]:
fig, axs = plt.subplots(ncols=4, nrows=2, figsize=(16, 8))
sc.pl.umap(adata, color=groupby,legend_fontsize=4, legend_fontoutline=2,size=4,legend_loc='on data',ax=axs[0,0],show=False)
sc.pl.umap(adata, color='CD4',legend_fontsize=4, legend_fontoutline=2,size=4,legend_loc='on data',ax=axs[0,1],show=False)
sc.pl.umap(adata, color='CD14',legend_fontsize=4, legend_fontoutline=2,size=4,legend_loc='on data',ax=axs[0,2],show=False)
sc.pl.umap(adata, color='FCGR3B',legend_fontsize=4, legend_fontoutline=2,size=4,legend_loc='on data',ax=axs[0,3],show=False)
#sc.pl.violin(adt,'CD45RA',groupby=groupby,show=False, ax=axs[0,3],size=0.5)
#TNFAIP3/CXCR4/DDIT4/FOSL2/IRS2/BTG1/ZFP36/PNRC1/CDKN1B/PLIN2/BHLHE40/PLAUR/ETS1/DUSP1
#SELL/TXNIP/IFITM2/B2M/HLA-C

sc.pl.violin(adata,'CXCR4',groupby=groupby,show=False, ax=axs[1,0],size=0.5)
sc.pl.violin(adata,'CSF2RB',groupby=groupby,show=False, ax=axs[1,1],size=0.5)
sc.pl.violin(adata,'IL16',groupby=groupby,show=False, ax=axs[1,2],size=0.5)
sc.pl.violin(adata,'IFNGR1',groupby=groupby,show=False, ax=axs[1,3],size=0.5)
plt.show()

In [ ]:
sc.pl.violin(adata,'CLC',groupby=groupby,size=0.5)

In [287]:
adata.obs[groupby].value_counts()

L2_leiden_scVI_0.5
3    767
4    543
1    349
5    339
7    291
6    267
2    233
0    178
8     43
Name: count, dtype: int64

In [288]:
# 特征CD25偏高 CCR3 CSF2RB
# Doublet识别
cell_dict = {'Doublet':['0',#细胞量少，Moncyte标记
                        '1',#NK、CD8T标记，
                        '2',#CD4T标记
                        '6',#Neu,
                        '8',#B
                       ],
             'Basophils':['3','4','5','7',
                        ],
            }

In [289]:
check_dict_duplicates(cell_dict)

'无重复'

In [290]:
# Generate new assignments
for i in cell_dict.keys():
    ind = pd.Series(adata.obs[groupby]).isin(cell_dict[i])
    adata.obs.loc[ind,'Celltype_L1_L2_Refine'] = i

In [294]:
(adata.obs['Celltype_L1_L2_Refine'].isna()).value_counts()

Celltype_L1_L2_Refine
False    3010
Name: count, dtype: int64

In [295]:
adata.obs.loc[adata.obs['receptor_type']=='TCR','Celltype_L1_L2_Refine']="Doublet"
adata.obs.loc[adata.obs['receptor_type_BCR']=='BCR','Celltype_L1_L2_Refine']="Doublet"

In [296]:
adata.obs['Celltype_L1_L2_Refine'].value_counts()

Celltype_L1_L2_Refine
Basophil    1769
Doublet     1241
Name: count, dtype: int64

In [297]:
adata = adata[adata.obs['Celltype_L1_L2_Refine'] != "Doublet",:]

In [298]:
adata.write(f"{obj_path}{celltype}/{celltype}_L2_Refine.h5ad",compression="gzip")

# 2. HSPC Cell Refine

In [9]:
celltype="HSPC"

In [10]:
adata = sc.read_h5ad(f"{obj_path}{celltype}/{celltype}_after_cluster_scVI.h5ad")
R_data_path = f"{obj_path}{celltype}"
sc.settings.figdir=f"{obj_path}{celltype}"

In [11]:
sc.pp.normalize_total(adata, target_sum=1e4)
sc.pp.log1p(adata)

In [12]:
del adata.obs['Celltype_L1_L2_Refine']

In [ ]:
sc.pl.umap(adata, color=['receptor_type','receptor_type_BCR'],
           frameon=False,
           legend_fontsize=4, legend_fontoutline=2, 
           size=4,
           legend_loc='on data')

In [14]:
pd.DataFrame(adata.obs[['L2_leiden_scVI_0.5', 'L2_leiden_scVI_1', 'L2_leiden_scVI_1.5', 'L2_leiden_scVI_2']]).to_csv(f'{R_data_path}/Level2_scVI_ForClustree.csv')
os.system(f'/home/liyanguo/anaconda3/envs/R/bin/Rscript ref_clustree.R {R_data_path} &')

0

Loading required package: ggraph
Loading required package: ggplot2


In [13]:
groupby = "L2_leiden_scVI_0.5"

In [16]:
xlsx = pd.ExcelWriter(f"{obj_path}{celltype}/{groupby}_freq_table.xlsx")
for group in ['SampleID', 'DonorID','scDblFinder.class','Phase',
              'Immune_All_High', 'Immune_All_Low', 'Adult_Human_Blood', 'Adult_Human_Bone_marrow',
              'AIFI_L1', 'AIFI_L2', 'AIFI_L3', 'predicted.celltype.l1', 'predicted.celltype.l2',
              'predicted.celltype.l3',
              'receptor_subtype', #TCR
              'VJ_1_v_call', 'VJ_1_j_call', 
              'VDJ_1_v_call', 'VDJ_1_j_call',
              'receptor_type_BCR', #BCR
              'VJ_1_v_call_BCR','VJ_1_j_call_BCR', 
              'VDJ_1_v_call_BCR', 'VDJ_1_j_call_BCR']:
    freq_table_multi = adata.obs.groupby([groupby, group]).size()
    pd.DataFrame(freq_table_multi).to_excel(xlsx,sheet_name=group)
xlsx.close()

In [ ]:
#数据簇间信息比较
fig, axs = plt.subplots(5, 2, figsize=(18, 12),constrained_layout=True)
plt.subplots_adjust(hspace=1,wspace=1)
sc.pl.violin(adata, keys='nCount_RNA', groupby=groupby, rotation=90,size=0.1,palette="pastel",linewidth=0,ax=axs[0,0], show=False)
sc.pl.violin(adata, keys='nFeature_RNA', groupby=groupby, rotation=90,size=0.1,palette="pastel",linewidth=0,ax=axs[0,1], show=False)
sc.pl.violin(adata, keys='log10GenesPerUMI', groupby=groupby, rotation=90,size=0.1,palette="pastel",linewidth=0,ax=axs[1,0], show=False)
sc.pl.violin(adata, keys='pct_counts_in_top_20_genes', groupby=groupby, rotation=90,size=0.1,palette="pastel",linewidth=0,ax=axs[1,1], show=False)

sc.pl.violin(adata, keys='percent_apop', groupby=groupby, rotation=90,size=0.1,palette="pastel",linewidth=0,ax=axs[2,0], show=False)
sc.pl.violin(adata, keys='percent_ribo', groupby=groupby, rotation=90,size=0.1,palette="pastel",linewidth=0,ax=axs[2,1], show=False)
sc.pl.violin(adata, keys='percent_ieg', groupby=groupby, rotation=90,size=0.1,palette="pastel",linewidth=0,ax=axs[3,0], show=False)
sc.pl.violin(adata, keys='percent_oxphos', groupby=groupby, rotation=90,size=0.1,palette="pastel",linewidth=0,ax=axs[3,1], show=False)
sc.pl.violin(adata, keys='percent_hemo', groupby=groupby, rotation=90,size=0.1,palette="pastel",linewidth=0,ax=axs[4,0], show=False)
sc.pl.violin(adata, keys='G2M.Score', groupby=groupby, rotation=90,size=0.1,palette="pastel",linewidth=0,ax=axs[4,1], show=False)

fig.tight_layout()
plt.savefig(f'{obj_path}{celltype}/{groupby}_scVI_L2_metadata.png')

In [14]:
sc.tl.dendrogram(adata,groupby=groupby,use_rep='X_scVI')

In [ ]:
#cosg差异
cosg.cosg(adata, key_added=f'cosg_{groupby}', groupby=groupby,
          mu=10,n_genes_user=100,remove_lowly_expressed=True,
         )
df_tmp = pd.DataFrame(adata.uns[f'cosg_{groupby}']['names'])
df_tmp.to_csv(f"{obj_path}{celltype}/cosg_{groupby}.csv")
#cosg差异作图
df_tmp=pd.DataFrame(adata.uns[f'cosg_{groupby}']['names'][:8,]).T
df_tmp=df_tmp.reindex(adata.uns['dendrogram_'+groupby]['categories_ordered'])
marker_genes_list={idx: list(row.values) for idx, row in df_tmp.iterrows()}
marker_genes_list = {k: v for k, v in marker_genes_list.items() if not any(isinstance(x, float) for x in v)}
sc.pl.dotplot(adata, marker_genes_list,
             groupby=groupby,
             dendrogram=True,
             swap_axes=False,
             standard_scale='var',
              save=f'cosg_{groupby}',
             cmap='Spectral_r')

In [20]:
os.system(f'nohup /home/liyanguo/anaconda3/envs/R/bin/Rscript DEG_ROC.R {R_data_path} {celltype} {groupby} &')

0

In [21]:
FeatureMatrix = pd.DataFrame(adata.uns[f'cosg_{groupby}']['names'])

In [22]:
genes_to_check ={
'HSC': ['HLF','NOG','PROM1','AVP','KLF2','CRHBP','HOPX','IDS','BST2','SPINK2'],
'MPP': ['C1QTNF4','SMIM24','SELL',],
'GMP': ['MPO','ELANE','AZU1','PRTN3','CALR','CFD','CTSG','MGST1','PRSS57'],
'NeuP': ['LYZ','LGALS1','CTSG','ANXA2'],
'MDP': ['SPIB','CD68','JCHAIN','IRF7','IRF8','PLD4','LILRA4','GZMB','CCDC50','SCT','UGCG'],
'EBMP': ['CLC','MS4A3','TPSAB1','MS4A2','LMO4','HDC'],
'CLP': ['DNTT','JCHAIN','IL7R','ADA','TRBC2','SATB1'],
'preB1': ['HMGB2','STMN1','CCND3','HMGN2',],
'preB2_3': ['CD9','CD79B','VPREB3','VPREB1','DNTT','ARPP21','AKAP12','CD79A','MZB1','MDM4'],
'preB3': ['GATA1','GATA2','TESPA1','CTNNBL1'],
'NK/Tp': ['CD96','IL32','KLRB1','TRAC','CD3E','TRBC1','CD3D','CCL5','CD7','NKG7'],
'MEP': ['APOC1','HBD','KLF1','BLVRB','TMEM14C','ITGA2B','MYC'],
'MKp1': ['GATA2','FCER1A','PBX1','CYTL1','SLC40A1'],
'MKp2': ['PPBP','PF4','NRGN','PLEK','RGS18','PDLIM1'],
'Eryp': ['HBB','HBA2','HBA1','AHSP','CA1','FAM178B',],
}
#Deciphering transcriptome alterations in bone marrow hematopoiesis at single-cell resolution in immune thrombocytopenia. Signal Transduct Target Ther 2022 Oct 7;7(1):347. PMID: 36202780

In [ ]:
sc.pl.dotplot(adata, genes_to_check,
             groupby=groupby,
             dendrogram=True,
             swap_axes=False,
             standard_scale='var',
              save=f'cosg_{groupby}',
             cmap='Spectral_r')

In [24]:
tfs_to_check ={
'HSC|MPP': ['AVP','KLF2','CRHBP','HOPX','BST2','SPINK2','MECOM','HOXA7','HOXA3','HESX1'],
'GMP|NeuP': ['CEBPA','ASCL2','CEBPD','CEBPE'],
'MDP': ['IRF7','SPIB','IRF4'],
'EBMP': ['POU2F2','GATA2','IKZF2','MEIS2','MITF'],
'CLP': ['MAFK','MEF2A',],
'preB1': ['E2F7','E2F8','E2F2'],
'preB2|3': ['BACH2','TCF3','FOXO1','EBF1','PAX5'],
'NK/Tp': ['RORA','EOMES','TBX21','BCL11B'],
'MEP|MKp1|MKp2|Eryp': ['GFI1B','GATA1','NFIB','HES5','PPARA'],
}
#Deciphering transcriptome alterations in bone marrow hematopoiesis at single-cell resolution in immune thrombocytopenia. Signal Transduct Target Ther 2022 Oct 7;7(1):347. PMID: 36202780

In [ ]:
sc.pl.dotplot(adata, tfs_to_check,
             groupby=groupby,
             dendrogram=True,
             swap_axes=False,
             standard_scale='var',
             cmap='Spectral_r')

In [ ]:
#基因集富集NK
sc.tl.score_genes(adata,['IL2RB','IRF8','JAK1','NKG7','PLEK','ZEB2','EOMES','GZMA','PRF1',])
sc.pl.umap(adata, color='score',legend_fontsize=4, legend_fontoutline=2,size=4,legend_loc='on data')

In [ ]:
fig, axs = plt.subplots(ncols=4, nrows=2, figsize=(16, 8))
sc.pl.umap(adata, color=groupby,legend_fontsize=4, legend_fontoutline=2,size=4,legend_loc='on data',ax=axs[0,0],show=False)
sc.pl.umap(adata, color='IL7R',legend_fontsize=4, legend_fontoutline=2,size=4,legend_loc='on data',ax=axs[0,1],show=False)
sc.pl.umap(adata, color='KIT',legend_fontsize=4, legend_fontoutline=2,size=4,legend_loc='on data',ax=axs[0,2],show=False)
sc.pl.umap(adata, color='MPL',legend_fontsize=4, legend_fontoutline=2,size=4,legend_loc='on data',ax=axs[0,3],show=False)

sc.pl.violin(adata,'SPI1',groupby=groupby,show=False, ax=axs[1,0],size=0.5)
sc.pl.violin(adata,'GATA2',groupby=groupby,show=False, ax=axs[1,1],size=0.5)
sc.pl.violin(adata,'CD38',groupby=groupby,show=False, ax=axs[1,2],size=0.5)
sc.pl.violin(adata,'CD34',groupby=groupby,show=False, ax=axs[1,3],size=0.5)
plt.show()
# PU.1/  CD71/ HBD MYB IL1B ITGA2B-low ALDH1A1 ST8SIA6 GMPR TPO MPL=CD110+ FLT3low GATA1 血红素代谢 CD45RA-

In [ ]:
sc.pl.violin(adata,'CXCR4',groupby=groupby)

In [15]:
#KEGG_HEMATOPOIETIC_cell_Lineage的富集查看

cell_dict = {
            'Mast':['1',],#KIT等
             
            'HSPC':['0','4','5',
                    '2',#KIT CD3
                    '3',#DNTT ,gene 和UMI低
                    '6',#DNTT
                    ],
            }
#CD7- 	CD34+ CD133+ SPI1 MYB MPL=CD110+ CSF3R=G-CSFR+  CD45RA-
#'Circulating MEPs'
#CD34+ CD38+ CD123- FLT3=CD135+ CD45RA-   CD38-low  AVP 
#'Circulating MkPs'#megakaryocyte  progenitor cells  PLEK

#CD7low CD34-  KIT+ CD133- FLT3=CD135+ PLD4  SPIB SPI1+(出骨髓的) CD45RA- IL7R- LMO4 MHCII+
#'Circulating MDPs'#common dendritic cell progenitors
#CCR7 RUNX2  JCHAIN   IGHM IGHD SOX4 GATA1- CD38- RORC

#CD7- 	CD34+  CD133+ KIT- FLT3=CD135+ PU.1/SPI1+(出骨髓的) MME CD45RA- IL7R-  ADA 
#'Circulating CLPs'
#CCR7  RUNX2  JCHAIN DNTT/tdt  IGHM IGHD SOX4 GATA1- CD38- LMO4

#Lin- CD7+ KIT+ CD96+ IL7R+ SELPLG+(CLP前往胸腺的因子) CD45RA+  SOX4-  FLT3=CD135-  CXCR4low SELL
#'Circulating NK/TPs'
#TCF7 IL32 ETS1（表明已经过了CLP阶段）  IL2RA=CD25+
#ILC2表达GATA3,但是需要KIT-/CRTH2+,所以不是
#ILC3需要RORC和AHR+，不是

# ATXN1（SCA1）
#T-restricted progenitors, circulating T progenitors, T-committed precursors PMID: 21251013, PMID: 17222572
#GMP 需要MPO
#https://www.nature.com/articles/s41467-022-29675-w    CD34+EPCR(PROCR)+(CD38/CD45RA)−   CD34+CD38−CD45RA− HSC    CD90+CD49f(ITGA6)+ HSC and CD90−CD49f− MPPs      CD90+CD49f+ and CD90−CD49f+ HSC
#https://www.cell.com/cell/fulltext/S0092-8674(24)00408-2#author-highlights-abstract MEP  AVP
#https://www.bdbiosciences.com/content/dam/bdb/marketing-documents/BD_Multiparametric_Immuno_Stem_Cells_AppNote.pdf
#https://link.springer.com/article/10.1007/s00018-018-2865-1#Sec3 NK ILC问题
# From stem cell to T cell: one route or many? 
# 血液中 HSC 的频率随昼夜节律振荡而波动，在光明开始后 5 小时达到峰值，在天黑后至少 5 小时达到峰值。我们在早上取样的（PMID: 18256599）
#  Exacting experiments to uncover the mechanisms of lymphoid progenitor homing from the BM to the thymus have been hampered by difficulties in identifying the relevant progenitor population

In [16]:
check_dict_duplicates(cell_dict)

'无重复'

In [17]:
# Generate new assignments
for i in cell_dict.keys():
    ind = pd.Series(adata.obs[groupby]).isin(cell_dict[i])
    adata.obs.loc[ind,'Celltype_L1_L2_Refine'] = i

In [18]:
adata.obs[groupby].value_counts()

L2_leiden_scVI_0.5
0    74
1    70
6    67
4    66
3    54
5    37
2    21
Name: count, dtype: int64

In [19]:
adata.obs['Celltype_L1_L2_Refine'].value_counts()

Celltype_L1_L2_Refine
HSPC    319
Mast     70
Name: count, dtype: int64

In [20]:
adata.obs.loc[adata.obs['receptor_type']=='TCR','Celltype_L1_L2_Refine']="Doublet"
adata.obs.loc[adata.obs['receptor_type_BCR']=='BCR','Celltype_L1_L2_Refine']="Doublet"

In [21]:
adata.obs['Celltype_L1_L2_Refine'].value_counts()

Celltype_L1_L2_Refine
HSPC       261
Doublet     70
Mast        58
Name: count, dtype: int64

In [22]:
adata = adata[adata.obs['Celltype_L1_L2_Refine'] != "Doublet",:]

In [23]:
adata.write(f"{obj_path}{celltype}/{celltype}_L2_Refine.h5ad",compression="gzip")

# 3. BCell Refine

In [220]:
celltype="B"

In [221]:
adata = sc.read_h5ad(f"{obj_path}{celltype}/{celltype}_after_cluster_scVI.h5ad")
R_data_path = f"{obj_path}{celltype}"
sc.settings.figdir=f"{obj_path}{celltype}"

In [222]:
sc.pp.normalize_total(adata, target_sum=1e4)
sc.pp.log1p(adata)

In [223]:
del adata.obs['Celltype_L1_L2_Refine']

In [ ]:
sc.pl.umap(adata, color=['receptor_type','chain_pairing_BCR'],
           frameon=False,
           legend_fontsize=4, legend_fontoutline=2, 
           size=4,
           legend_loc='on data')

In [225]:
pd.DataFrame(adata.obs[['L2_leiden_scVI_0.5', 'L2_leiden_scVI_1', 'L2_leiden_scVI_1.5', 'L2_leiden_scVI_2']]).to_csv(f'{R_data_path}/Level2_scVI_ForClustree.csv')
os.system(f'/home/liyanguo/anaconda3/envs/R/bin/Rscript ref_clustree.R {R_data_path} &')

0

In [226]:
groupby = "L2_leiden_scVI_0.5"

In [227]:
xlsx = pd.ExcelWriter(f"{obj_path}{celltype}/{groupby}_freq_table.xlsx")
for group in ['SampleID', 'DonorID','scDblFinder.class','Phase',
              'Immune_All_High', 'Immune_All_Low', 'Adult_Human_Blood', 'Adult_Human_Bone_marrow',
              'AIFI_L1', 'AIFI_L2', 'AIFI_L3', 'predicted.celltype.l1', 'predicted.celltype.l2',
              'predicted.celltype.l3',
              'receptor_subtype', #TCR
              'VJ_1_v_call', 'VJ_1_j_call', 
              'VDJ_1_v_call', 'VDJ_1_j_call',
              'receptor_type_BCR', #BCR
              'VJ_1_v_call_BCR','VJ_1_j_call_BCR', 
              'VDJ_1_v_call_BCR', 'VDJ_1_j_call_BCR']:
    freq_table_multi = adata.obs.groupby([groupby, group]).size()
    pd.DataFrame(freq_table_multi).to_excel(xlsx,sheet_name=group)
xlsx.close()

Loading required package: ggraph
Loading required package: ggplot2


In [ ]:
#数据簇间信息比较
fig, axs = plt.subplots(5, 2, figsize=(18, 12),constrained_layout=True)
sc.pl.violin(adata, keys='nCount_RNA', groupby=groupby, rotation=90,size=0.1,palette="pastel",linewidth=0,ax=axs[0,0], show=False)
sc.pl.violin(adata, keys='nFeature_RNA', groupby=groupby, rotation=90,size=0.1,palette="pastel",linewidth=0,ax=axs[0,1], show=False)
sc.pl.violin(adata, keys='log10GenesPerUMI', groupby=groupby, rotation=90,size=0.1,palette="pastel",linewidth=0,ax=axs[1,0], show=False)
sc.pl.violin(adata, keys='pct_counts_in_top_20_genes', groupby=groupby, rotation=90,size=0.1,palette="pastel",linewidth=0,ax=axs[1,1], show=False)

sc.pl.violin(adata, keys='percent_apop', groupby=groupby, rotation=90,size=0.1,palette="pastel",linewidth=0,ax=axs[2,0], show=False)
sc.pl.violin(adata, keys='percent_ribo', groupby=groupby, rotation=90,size=0.1,palette="pastel",linewidth=0,ax=axs[2,1], show=False)
sc.pl.violin(adata, keys='percent_ieg', groupby=groupby, rotation=90,size=0.1,palette="pastel",linewidth=0,ax=axs[3,0], show=False)
sc.pl.violin(adata, keys='percent_oxphos', groupby=groupby, rotation=90,size=0.1,palette="pastel",linewidth=0,ax=axs[3,1], show=False)
sc.pl.violin(adata, keys='percent_hemo', groupby=groupby, rotation=90,size=0.1,palette="pastel",linewidth=0,ax=axs[4,0], show=False)
sc.pl.violin(adata, keys='G2M.Score', groupby=groupby, rotation=90,size=0.1,palette="pastel",linewidth=0,ax=axs[4,1], show=False)

fig.tight_layout()
plt.savefig(f'{obj_path}{celltype}/{groupby}_scVI_L2_metadata.png')

In [229]:
sc.tl.dendrogram(adata,groupby=groupby,use_rep='X_scVI')

In [ ]:
#cosg差异
cosg.cosg(adata, key_added=f'cosg_{groupby}', groupby=groupby,
          mu=10,n_genes_user=100,remove_lowly_expressed=True,
         )
df_tmp = pd.DataFrame(adata.uns[f'cosg_{groupby}']['names'])
df_tmp.to_csv(f"{obj_path}{celltype}/cosg_{groupby}.csv")
#cosg差异作图
df_tmp=pd.DataFrame(adata.uns[f'cosg_{groupby}']['names'][:8,]).T
df_tmp=df_tmp.reindex(adata.uns['dendrogram_'+groupby]['categories_ordered'])
marker_genes_list={idx: list(row.values) for idx, row in df_tmp.iterrows()}
marker_genes_list = {k: v for k, v in marker_genes_list.items() if not any(isinstance(x, float) for x in v)}
sc.pl.dotplot(adata, marker_genes_list,
             groupby=groupby,
             dendrogram=True,
             swap_axes=False,
             standard_scale='var',
              save=f'cosg_{groupby}',
             cmap='Spectral_r')

In [239]:
os.system(f'nohup /home/liyanguo/anaconda3/envs/R/bin/Rscript DEG_ROC.R {R_data_path} {celltype} {groupby} &')

0

In [240]:
FeatureMatrix = pd.DataFrame(adata.uns[f'cosg_{groupby}']['names'])

[1] "Sample 100000 cells per group('L2_leiden_scVI_0.5')!"


Normalizing layer: counts
Performing log-normalization
0%   10   20   30   40   50   60   70   80   90   100%
[----|----|----|----|----|----|----|----|----|----|
**************************************************|
Calculating cluster 0
Calculating cluster 1
Calculating cluster 2
Calculating cluster 3
Calculating cluster 4
Calculating cluster 5
Calculating cluster 6


In [ ]:
%%R -i FeatureMatrix,R_data_path,groupby
source('GetEnrich.R')
GetEnrich(FeatureMatrix,plot_term_number = 10,R_data_path=R_data_path,groupby=groupby,
          gmtfile="/localdisk/immune/Marker/msigdb_v2024.1.Hs_GMTs/c2.cp.kegg_legacy.v2024.1.Hs.symbols.gmt")
#"/localdisk/immune/Marker/msigdb_v2024.1.Hs_GMTs/h.all.v2024.1.Hs.symbols.gmt"
#"/localdisk/immune/Marker/Immunity_PBMC.gmt"
#"/localdisk/immune/Marker/sepsis_PBMC.gmt"
#"/localdisk/immune/Marker/msigdb_v2024.1.Hs_GMTs/c5.go.bp.v2024.1.Hs.symbols.gmt"
#"/localdisk/immune/Marker/msigdb_v2024.1.Hs_GMTs/c2.cp.kegg_legacy.v2024.1.Hs.symbols.gmt"

In [ ]:
fig, axs = plt.subplots(ncols=4, nrows=2, figsize=(16, 8))
sc.pl.umap(adata, color=groupby,legend_fontsize=4, legend_fontoutline=2,size=4,legend_loc='on data',ax=axs[0,0],show=False)
sc.pl.umap(adata, color='CD3D',legend_fontsize=4, legend_fontoutline=2,size=4,legend_loc='on data',ax=axs[0,1],show=False)
sc.pl.umap(adata, color='SPON2',legend_fontsize=4, legend_fontoutline=2,size=4,legend_loc='on data',ax=axs[0,2],show=False)
sc.pl.umap(adata, color='CD14',legend_fontsize=4, legend_fontoutline=2,size=4,legend_loc='on data',ax=axs[0,3],show=False)

sc.pl.violin(adata,'IGHM',groupby=groupby,show=False, ax=axs[1,0],size=0)
sc.pl.violin(adata,'IGHD',groupby=groupby,show=False, ax=axs[1,1],size=0)
sc.pl.violin(adata,'CSF3R',groupby=groupby,show=False, ax=axs[1,2],size=0)
sc.pl.violin(adata,'CD14',groupby=groupby,show=False, ax=axs[1,3],size=0)
plt.show()

In [ ]:
sc.pl.violin(adata,'TCF4',groupby=groupby,size=0)

In [241]:
cell_dict = {
             'Doublet':['0',#TNK, scDBFinder同样提示高Doublets
                        '1',#T, scDBFinder同样提示高Doublets
                        '6',#Monocyte, scDBFinder同样提示高Doublets
                        '5',#Neu, scDBFinder同样提示较高Doublets
                       ],
             'B cells':['2','3','4','8','9'],
             'Plamsa cells':['7',],
            }

#https://www.cellsignal.cn/pathways/immune-cell-markers-human
#https://www.abcam.cn/primary-antibodies/b-cells-basic-immunophenotyping
#https://www.bdbiosciences.com/zh-cn/learn/research/immunology/b-cells#B-Cell-Immunophenotyping
#https://ashpublications.org/blood/article/114/25/5173/26511/An-in-vitro-model-of-differentiation-of-memory-B
#浆细胞、浆母细胞https://www.sohu.com/a/234622412_610701;
#https://zhuanlan.zhihu.com/p/56657003
#https://www.biocompare.com/Editorial-Articles/570578-A-Guide-to-B-Cell-Markers/
# https://www.cell.com/trends/immunology/fulltext/S1471-4906(22)00003-5#secst0040
# https://www.nature.com/articles/s41577-024-00991-0

# IgD的出现，意味着B细胞从原先耐受抗原的细胞，开始逐渐转变为可对抗原产生反应的B细胞了，
# 所以从T1、T2、T3开始IgD逐渐增高，到Naive达到高峰。当Naive刚变成Memory B时，IgD仍保留着，所以此时叫Unswitched（未转换），
# 只有Memory B的表面Ig从IgD转变为IgG、IgA或IgE时，才叫Switched Memory B
# ，就可以进一步向Plasmablast（浆母细胞）转换，开始大量分泌抗体。

In [242]:
adata.obs[groupby].value_counts()

L2_leiden_scVI_0.5
9    10540
5     8217
8     8055
3     6911
4     5521
0     4946
1     2788
7     2219
6      741
2      484
Name: count, dtype: int64

In [243]:
check_dict_duplicates(cell_dict)

'无重复'

In [247]:
# Generate new assignments
for i in cell_dict.keys():
    ind = pd.Series(adata.obs[groupby]).isin(cell_dict[i])
    adata.obs.loc[ind,'Celltype_L1_L2_Refine'] = i

In [248]:
(adata.obs['Celltype_L1_L2_Refine'].isna()).value_counts()

Celltype_L1_L2_Refine
False    50422
Name: count, dtype: int64

In [249]:
adata.obs['Celltype_L1_L2_Refine'].value_counts()

Celltype_L1_L2_Refine
B          31511
Doublet    16692
Plamsa      2219
Name: count, dtype: int64

In [250]:
adata.obs.loc[adata.obs['receptor_type']=='TCR','Celltype_L1_L2_Refine']="Doublet"

In [251]:
adata.obs['Celltype_L1_L2_Refine'].value_counts()

Celltype_L1_L2_Refine
B          28220
Doublet    20360
Plamsa      1842
Name: count, dtype: int64

In [252]:
adata = adata[adata.obs['Celltype_L1_L2_Refine'] != "Doublet",:]

In [253]:
adata.write(f"{obj_path}{celltype}/{celltype}_L2_Refine.h5ad",compression="gzip")

Calculating cluster 8
Calculating cluster 9


[1] "Done!"


# 4. CD4TCell Refine

In [116]:
celltype="CD4T"

In [117]:
adata = sc.read_h5ad(f"{obj_path}{celltype}/{celltype}_after_cluster_scVI.h5ad")
R_data_path = f"{obj_path}{celltype}"
sc.settings.figdir=f"{obj_path}{celltype}"

In [118]:
sc.pp.normalize_total(adata, target_sum=1e4)
sc.pp.log1p(adata)

In [119]:
del adata.obs['Celltype_L1_L2_Refine']

In [ ]:
sc.pl.umap(adata, color=['receptor_type','receptor_type_BCR'],
           frameon=False,
           legend_fontsize=4, legend_fontoutline=2, 
           size=4,
           legend_loc='on data')

In [121]:
pd.DataFrame(adata.obs[['L2_leiden_scVI_0.5', 'L2_leiden_scVI_1', 'L2_leiden_scVI_1.5', 'L2_leiden_scVI_2']]).to_csv(f'{R_data_path}/Level2_scVI_ForClustree.csv')
os.system(f'/home/liyanguo/anaconda3/envs/R/bin/Rscript ref_clustree.R {R_data_path} &')

0

In [122]:
groupby = "L2_leiden_scVI_1"

Loading required package: ggraph
Loading required package: ggplot2


In [91]:
xlsx = pd.ExcelWriter(f"{obj_path}{celltype}/{groupby}_freq_table.xlsx")
for group in ['SampleID', 'DonorID','scDblFinder.class','Phase',
              'Immune_All_High', 'Immune_All_Low', 'Adult_Human_Blood', 'Adult_Human_Bone_marrow',
              'AIFI_L1', 'AIFI_L2', 'AIFI_L3', 'predicted.celltype.l1', 'predicted.celltype.l2',
              'predicted.celltype.l3',
              'receptor_subtype', #TCR
              'VJ_1_v_call', 'VJ_1_j_call', 
              'VDJ_1_v_call', 'VDJ_1_j_call',
              'receptor_type_BCR', #BCR
              'VJ_1_v_call_BCR','VJ_1_j_call_BCR', 
              'VDJ_1_v_call_BCR', 'VDJ_1_j_call_BCR']:
    freq_table_multi = adata.obs.groupby([groupby, group]).size()
    pd.DataFrame(freq_table_multi).to_excel(xlsx,sheet_name=group)
xlsx.close()

Loading required package: ggraph
Loading required package: ggplot2


In [ ]:
#数据簇间信息比较
fig, axs = plt.subplots(5, 2, figsize=(18, 12),constrained_layout=True)
plt.subplots_adjust(hspace=1,wspace=1)
sc.pl.violin(adata, keys='nCount_RNA', groupby=groupby, rotation=90,size=0.1,palette="pastel",linewidth=0,ax=axs[0,0], show=False)
sc.pl.violin(adata, keys='nFeature_RNA', groupby=groupby, rotation=90,size=0.1,palette="pastel",linewidth=0,ax=axs[0,1], show=False)
sc.pl.violin(adata, keys='log10GenesPerUMI', groupby=groupby, rotation=90,size=0.1,palette="pastel",linewidth=0,ax=axs[1,0], show=False)
sc.pl.violin(adata, keys='pct_counts_in_top_20_genes', groupby=groupby, rotation=90,size=0.1,palette="pastel",linewidth=0,ax=axs[1,1], show=False)

sc.pl.violin(adata, keys='percent_apop', groupby=groupby, rotation=90,size=0.1,palette="pastel",linewidth=0,ax=axs[2,0], show=False)
sc.pl.violin(adata, keys='percent_ribo', groupby=groupby, rotation=90,size=0.1,palette="pastel",linewidth=0,ax=axs[2,1], show=False)
sc.pl.violin(adata, keys='percent_ieg', groupby=groupby, rotation=90,size=0.1,palette="pastel",linewidth=0,ax=axs[3,0], show=False)
sc.pl.violin(adata, keys='percent_oxphos', groupby=groupby, rotation=90,size=0.1,palette="pastel",linewidth=0,ax=axs[3,1], show=False)
sc.pl.violin(adata, keys='percent_hemo', groupby=groupby, rotation=90,size=0.1,palette="pastel",linewidth=0,ax=axs[4,0], show=False)
sc.pl.violin(adata, keys='G2M.Score', groupby=groupby, rotation=90,size=0.1,palette="pastel",linewidth=0,ax=axs[4,1], show=False)

fig.tight_layout()
plt.savefig(f'{obj_path}{celltype}/{groupby}_scVI_L2_metadata.png')

In [93]:
adata.obs.groupby([groupby,'receptor_type']).size()

L2_leiden_scVI_1  receptor_type
0                 TCR                606
1                 TCR              12348
2                 TCR              15470
3                 TCR              10394
4                 TCR               9214
5                 TCR              22563
6                 TCR               8582
7                 TCR               9038
8                 TCR               8382
9                 TCR               2536
10                TCR              12160
11                TCR                667
12                TCR               4497
13                TCR              16902
14                TCR                408
dtype: int64

In [ ]:
adata.obs['cluster_dummy'] = adata.obs[groupby] == adata.obs[groupby].cat.categories[11]
sc.pl.umap(adata, color='cluster_dummy',legend_fontsize=4, legend_fontoutline=2,size=4,legend_loc='on data')

In [95]:
sc.tl.dendrogram(adata,groupby=groupby,use_rep='X_scVI')

In [ ]:
#cosg差异
cosg.cosg(adata, key_added=f'cosg_{groupby}', groupby=groupby,
          mu=10,n_genes_user=100,remove_lowly_expressed=True,
         )
df_tmp = pd.DataFrame(adata.uns[f'cosg_{groupby}']['names'])
df_tmp.to_csv(f"{obj_path}{celltype}/cosg_{groupby}.csv")
#cosg差异作图
df_tmp=pd.DataFrame(adata.uns[f'cosg_{groupby}']['names'][:8,]).T
df_tmp=df_tmp.reindex(adata.uns['dendrogram_'+groupby]['categories_ordered'])
marker_genes_list={idx: list(row.values) for idx, row in df_tmp.iterrows()}
marker_genes_list = {k: v for k, v in marker_genes_list.items() if not any(isinstance(x, float) for x in v)}
sc.pl.dotplot(adata, marker_genes_list,
             groupby=groupby,
             dendrogram=True,
             swap_axes=False,
             standard_scale='var',
             cmap='Spectral_r')

In [97]:
FeatureMatrix = pd.DataFrame(adata.uns[f'cosg_{groupby}']['names'])

In [ ]:
%%R -i FeatureMatrix,R_data_path,groupby
source('GetEnrich.R')
GetEnrich(FeatureMatrix,plot_term_number = 10,R_data_path=R_data_path,groupby=groupby,
          gmtfile="/localdisk/immune/Marker/Immunity_PBMC.gmt")
#"/localdisk/immune/Marker/Immunity_PBMC.gmt"
#"/localdisk/immune/Marker/sepsis_PBMC.gmt"
#"/localdisk/immune/Marker/msigdb_v2024.1.Hs_GMTs/c5.go.bp.v2024.1.Hs.symbols.gmt"
#"/localdisk/immune/Marker/msigdb_v2024.1.Hs_GMTs/c2.cp.kegg_legacy.v2024.1.Hs.symbols.gmt"

In [ ]:
fig, axs = plt.subplots(ncols=4, nrows=2, figsize=(16, 8))
sc.pl.umap(adata, color=groupby,legend_fontsize=4, legend_fontoutline=2,size=1,legend_loc='on data',ax=axs[0,0],show=False)
sc.pl.umap(adata, color='CD14',legend_fontsize=4, legend_fontoutline=2,size=4,legend_loc='on data',ax=axs[0,1],show=False)
sc.pl.umap(adata, color='CCR4',legend_fontsize=4, legend_fontoutline=2,size=4,legend_loc='on data',ax=axs[0,2],show=False)
sc.pl.umap(adata, color='CSF3R',legend_fontsize=4, legend_fontoutline=2,size=4,legend_loc='on data',ax=axs[0,3],show=False)
sc.pl.umap(adata, color='CTLA4',legend_fontsize=4, legend_fontoutline=2,size=4,legend_loc='on data',ax=axs[1,0],show=False)
sc.pl.umap(adata, color='SPON2',legend_fontsize=4, legend_fontoutline=2,size=4,legend_loc='on data',ax=axs[1,1],show=False)
sc.pl.umap(adata, color='FOXP3',legend_fontsize=4, legend_fontoutline=2,size=4,legend_loc='on data',ax=axs[1,2],show=False)
sc.pl.umap(adata, color='CCR6',legend_fontsize=4, legend_fontoutline=2,size=4,legend_loc='on data',ax=axs[1,3],show=False)
plt.show()

In [123]:
cell_dict = {'Doublet':['0',#Mono + scDBFinder
                        '12',#Neu + scDBFinder
                        '14',#！非双细胞，低质量细胞，缺乏CD3D、CD7表达
                       ],
             
             'CD4+ T cells':['1','2','3','4','6','7','8','9','10','13',
                       '5',#CTL
                       '11',#CD8B+ CD4- Dn T特征
                      ],
            }

In [124]:
adata.obs[groupby].value_counts()

L2_leiden_scVI_1
5     31129
13    21280
2     19844
1     14573
10    13756
4     12922
3     12485
7     10665
8     10372
6     10359
12     5991
9      2724
14     1719
11      752
0       741
Name: count, dtype: int64

In [125]:
check_dict_duplicates(cell_dict)

'无重复'

In [126]:
# Generate new assignments
for i in cell_dict.keys():
    ind = pd.Series(adata.obs[groupby]).isin(cell_dict[i])
    adata.obs.loc[ind,'Celltype_L1_L2_Refine'] = i

In [127]:
(adata.obs['Celltype_L1_L2_Refine'].isna()).value_counts()

Celltype_L1_L2_Refine
False    169312
Name: count, dtype: int64

In [128]:
adata.obs['Celltype_L1_L2_Refine'].value_counts()

Celltype_L1_L2_Refine
CD4+ T     160861
Doublet      8451
Name: count, dtype: int64

In [129]:
adata.obs.loc[adata.obs['receptor_type_BCR']=='BCR','Celltype_L1_L2_Refine']="Doublet"

In [130]:
adata.obs['Celltype_L1_L2_Refine'].value_counts()

Celltype_L1_L2_Refine
CD4+ T     158067
Doublet     11245
Name: count, dtype: int64

In [131]:
adata = adata[adata.obs['Celltype_L1_L2_Refine'] != "Doublet",:]

In [132]:
adata.write(f"{obj_path}{celltype}/{celltype}_L2_Refine.h5ad",compression="gzip")

# 5. CD8TCell Refine

In [162]:
celltype="CD8T"

In [163]:
adata = sc.read_h5ad(f"{obj_path}{celltype}/{celltype}_after_cluster_scVI.h5ad")
R_data_path = f"{obj_path}{celltype}"
sc.settings.figdir=f"{obj_path}{celltype}"

In [164]:
sc.pp.normalize_total(adata, target_sum=1e4)
sc.pp.log1p(adata)

In [165]:
del adata.obs['Celltype_L1_L2_Refine']

In [ ]:
sc.pl.umap(adata, color=['receptor_type','receptor_type_BCR'],
           frameon=False,
           legend_fontsize=4, legend_fontoutline=2, 
           size=4,
           legend_loc='on data')

In [167]:
groupby = "L2_leiden_scVI_1"

In [168]:
xlsx = pd.ExcelWriter(f"{obj_path}{celltype}/{groupby}_freq_table.xlsx")
for group in ['SampleID', 'DonorID','scDblFinder.class','Phase',
              'Immune_All_High', 'Immune_All_Low', 'Adult_Human_Blood', 'Adult_Human_Bone_marrow',
              'AIFI_L1', 'AIFI_L2', 'AIFI_L3', 'predicted.celltype.l1', 'predicted.celltype.l2',
              'predicted.celltype.l3',
              'receptor_subtype', #TCR
              'VJ_1_v_call', 'VJ_1_j_call', 
              'VDJ_1_v_call', 'VDJ_1_j_call',
              'receptor_type_BCR', #BCR
              'VJ_1_v_call_BCR','VJ_1_j_call_BCR', 
              'VDJ_1_v_call_BCR', 'VDJ_1_j_call_BCR']:
    freq_table_multi = adata.obs.groupby([groupby, group]).size()
    pd.DataFrame(freq_table_multi).to_excel(xlsx,sheet_name=group)
xlsx.close()

In [ ]:
#数据簇间信息比较
fig, axs = plt.subplots(5, 2, figsize=(18, 12),constrained_layout=True)
plt.subplots_adjust(hspace=1,wspace=1)
sc.pl.violin(adata, keys='nCount_RNA', groupby=groupby, rotation=90,size=0.1,palette="pastel",linewidth=0,ax=axs[0,0], show=False)
sc.pl.violin(adata, keys='nFeature_RNA', groupby=groupby, rotation=90,size=0.1,palette="pastel",linewidth=0,ax=axs[0,1], show=False)
sc.pl.violin(adata, keys='log10GenesPerUMI', groupby=groupby, rotation=90,size=0.1,palette="pastel",linewidth=0,ax=axs[1,0], show=False)
sc.pl.violin(adata, keys='pct_counts_in_top_20_genes', groupby=groupby, rotation=90,size=0.1,palette="pastel",linewidth=0,ax=axs[1,1], show=False)

sc.pl.violin(adata, keys='percent_apop', groupby=groupby, rotation=90,size=0.1,palette="pastel",linewidth=0,ax=axs[2,0], show=False)
sc.pl.violin(adata, keys='percent_ribo', groupby=groupby, rotation=90,size=0.1,palette="pastel",linewidth=0,ax=axs[2,1], show=False)
sc.pl.violin(adata, keys='percent_ieg', groupby=groupby, rotation=90,size=0.1,palette="pastel",linewidth=0,ax=axs[3,0], show=False)
sc.pl.violin(adata, keys='percent_oxphos', groupby=groupby, rotation=90,size=0.1,palette="pastel",linewidth=0,ax=axs[3,1], show=False)
sc.pl.violin(adata, keys='percent_hemo', groupby=groupby, rotation=90,size=0.1,palette="pastel",linewidth=0,ax=axs[4,0], show=False)
sc.pl.violin(adata, keys='G2M.Score', groupby=groupby, rotation=90,size=0.1,palette="pastel",linewidth=0,ax=axs[4,1], show=False)

fig.tight_layout()
plt.savefig(f'{obj_path}{celltype}/{groupby}_scVI_L2_metadata.png')

In [170]:
pd.DataFrame(adata.obs[['L2_leiden_scVI_0.5', 'L2_leiden_scVI_1', 'L2_leiden_scVI_1.5', 'L2_leiden_scVI_2']]).to_csv(f'{R_data_path}/Level2_scVI_ForClustree.csv')
os.system(f'/home/liyanguo/anaconda3/envs/R/bin/Rscript ref_clustree.R {R_data_path} &')

0

In [171]:
sc.tl.dendrogram(adata,groupby=groupby,use_rep='X_scVI')

In [ ]:
#cosg差异
cosg.cosg(adata, key_added=f'cosg_{groupby}', groupby=groupby,
          mu=10,n_genes_user=100,remove_lowly_expressed=True,
         )
df_tmp = pd.DataFrame(adata.uns[f'cosg_{groupby}']['names'])
df_tmp.to_csv(f"{obj_path}{celltype}/cosg_{groupby}.csv")
#cosg差异作图
df_tmp=pd.DataFrame(adata.uns[f'cosg_{groupby}']['names'][:8,]).T
df_tmp=df_tmp.reindex(adata.uns['dendrogram_'+groupby]['categories_ordered'])
marker_genes_list={idx: list(row.values) for idx, row in df_tmp.iterrows()}
marker_genes_list = {k: v for k, v in marker_genes_list.items() if not any(isinstance(x, float) for x in v)}
sc.pl.dotplot(adata, marker_genes_list,
             groupby=groupby,
             dendrogram=True,
             swap_axes=False,
             standard_scale='var',
              save=f'cosg_{groupby}',
             cmap='Spectral_r')

In [173]:
adata.obs.groupby([groupby,'receptor_type']).size()

L2_leiden_scVI_1  receptor_type
0                 TCR              18913
1                 TCR               3864
2                 TCR               7949
3                 TCR               1471
4                 TCR               7923
5                 TCR               8664
6                 TCR               9960
7                 TCR               3788
8                 TCR               9726
9                 TCR               2781
10                TCR               7237
11                TCR               8960
12                TCR               8952
13                TCR                404
14                TCR                366
dtype: int64

In [ ]:
#基因集富集NK
sc.tl.score_genes(adata,['IL2RB','IRF8','JAK1','NKG7','PLEK','ZEB2','EOMES','GZMA','PRF1',])
sc.pl.umap(adata, color='score',legend_fontsize=4, legend_fontoutline=2,size=4,legend_loc='on data')

In [ ]:
fig, axs = plt.subplots(ncols=4, nrows=2, figsize=(16, 8))
sc.pl.umap(adata, color=groupby,legend_fontsize=4, legend_fontoutline=2,size=4,legend_loc='on data',ax=axs[0,0],show=False)
sc.pl.umap(adata, color='CD8A',legend_fontsize=4, legend_fontoutline=2,size=4,legend_loc='on data',ax=axs[0,1],show=False)
sc.pl.umap(adata, color='CD4',legend_fontsize=4, legend_fontoutline=2,size=4,legend_loc='on data',ax=axs[0,2],show=False)
sc.pl.umap(adata, color='GZMK',legend_fontsize=4, legend_fontoutline=2,size=4,legend_loc='on data',ax=axs[0,3],show=False)

sc.pl.violin(adata,'KLRG1',groupby=groupby,show=False, ax=axs[1,0],size=0)
sc.pl.violin(adata,'PRF1',groupby=groupby,show=False, ax=axs[1,1],size=0)
sc.pl.violin(adata,'FCGR3B',groupby=groupby,show=False, ax=axs[1,2],size=0)
sc.pl.violin(adata,'SPON2',groupby=groupby,show=False, ax=axs[1,3],size=0)
plt.show()

In [176]:
#naive (
#central memory CD45RO and high levels of TCF7 and LEF1 transcripts) CD27+ CD45RA− 
#effector memory T (Tem) CD8+ subpopulations that expressed either granzyme K or granzyme B   CD27− CD45RA−

cell_dict = {'Doublet':['14',#Mono + scDblFinder
                       '1',#Neu + scDblFinder
                       ],

             'Non-MAIT/NKT CD8+ T cells':['0','2','3','4','5','6','7','8','9','10','11','12','13'],#
            }

#注意: 此套数据，没有CD56, CITE-seq可以很好区分NK和CD8

In [177]:
adata.obs[groupby].value_counts()

L2_leiden_scVI_1
0     22164
8     11734
6     11715
11    11326
5     11123
4      9614
2      9446
12     9304
10     8580
1      5164
7      4579
9      4150
3      1904
13      450
14      447
Name: count, dtype: int64

In [178]:
check_dict_duplicates(cell_dict)

'无重复'

In [179]:
# Generate new assignments
for i in cell_dict.keys():
    ind = pd.Series(adata.obs[groupby]).isin(cell_dict[i])
    adata.obs.loc[ind,'Celltype_L1_L2_Refine'] = i

In [180]:
(adata.obs['Celltype_L1_L2_Refine'].isna()).value_counts()

Celltype_L1_L2_Refine
False    121700
Name: count, dtype: int64

In [181]:
adata.obs['Celltype_L1_L2_Refine'].value_counts()

Celltype_L1_L2_Refine
CD8+ T     116089
Doublet      5611
Name: count, dtype: int64

In [182]:
adata.obs.loc[adata.obs['receptor_type_BCR']=='BCR','Celltype_L1_L2_Refine']="Doublet"

In [183]:
adata.obs['Celltype_L1_L2_Refine'].value_counts()

Celltype_L1_L2_Refine
CD8+ T     114555
Doublet      7145
Name: count, dtype: int64

In [184]:
adata = adata[adata.obs['Celltype_L1_L2_Refine'] != "Doublet",:]

In [185]:
adata.write(f"{obj_path}{celltype}/{celltype}_L2_Refine.h5ad",compression="gzip")

# 6. MAITCell Refine

In [299]:
celltype="MAIT"

In [300]:
adata = sc.read_h5ad(f"{obj_path}{celltype}/{celltype}_after_cluster_scVI.h5ad")
R_data_path = f"{obj_path}{celltype}"
sc.settings.figdir=f"{obj_path}{celltype}"

In [301]:
sc.pp.normalize_total(adata, target_sum=1e4)
sc.pp.log1p(adata)

In [342]:
del adata.obs['Celltype_L1_L2_Refine']

In [ ]:
sc.pl.umap(adata, color=['receptor_type','receptor_type_BCR'],
           frameon=False,
           legend_fontsize=4, legend_fontoutline=2, 
           size=4,
           legend_loc='on data')

In [304]:
pd.DataFrame(adata.obs[['L2_leiden_scVI_0.5', 'L2_leiden_scVI_1', 'L2_leiden_scVI_1.5', 'L2_leiden_scVI_2']]).to_csv(f'{R_data_path}/Level2_scVI_ForClustree.csv')
os.system(f'/home/liyanguo/anaconda3/envs/R/bin/Rscript ref_clustree.R {R_data_path} &')

0

In [323]:
groupby = "L2_leiden_scVI_1"

In [324]:
xlsx = pd.ExcelWriter(f"{obj_path}{celltype}/{groupby}_freq_table.xlsx")
for group in ['SampleID', 'DonorID','scDblFinder.class','Phase',
              'Immune_All_High', 'Immune_All_Low', 'Adult_Human_Blood', 'Adult_Human_Bone_marrow',
              'AIFI_L1', 'AIFI_L2', 'AIFI_L3', 'predicted.celltype.l1', 'predicted.celltype.l2',
              'predicted.celltype.l3',
              'receptor_subtype', #TCR
              'VJ_1_v_call', 'VJ_1_j_call', 
              'VDJ_1_v_call', 'VDJ_1_j_call',
              'receptor_type_BCR', #BCR
              'VJ_1_v_call_BCR','VJ_1_j_call_BCR', 
              'VDJ_1_v_call_BCR', 'VDJ_1_j_call_BCR']:
    freq_table_multi = adata.obs.groupby([groupby, group]).size()
    pd.DataFrame(freq_table_multi).to_excel(xlsx,sheet_name=group)
xlsx.close()

In [ ]:
#数据簇间信息比较
fig, axs = plt.subplots(5, 2, figsize=(18, 12),constrained_layout=True)
plt.subplots_adjust(hspace=1,wspace=1)
sc.pl.violin(adata, keys='nCount_RNA', groupby=groupby, rotation=90,size=0.1,palette="pastel",linewidth=0,ax=axs[0,0], show=False)
sc.pl.violin(adata, keys='nFeature_RNA', groupby=groupby, rotation=90,size=0.1,palette="pastel",linewidth=0,ax=axs[0,1], show=False)
sc.pl.violin(adata, keys='log10GenesPerUMI', groupby=groupby, rotation=90,size=0.1,palette="pastel",linewidth=0,ax=axs[1,0], show=False)
sc.pl.violin(adata, keys='pct_counts_in_top_20_genes', groupby=groupby, rotation=90,size=0.1,palette="pastel",linewidth=0,ax=axs[1,1], show=False)

sc.pl.violin(adata, keys='percent_apop', groupby=groupby, rotation=90,size=0.1,palette="pastel",linewidth=0,ax=axs[2,0], show=False)
sc.pl.violin(adata, keys='percent_ribo', groupby=groupby, rotation=90,size=0.1,palette="pastel",linewidth=0,ax=axs[2,1], show=False)
sc.pl.violin(adata, keys='percent_ieg', groupby=groupby, rotation=90,size=0.1,palette="pastel",linewidth=0,ax=axs[3,0], show=False)
sc.pl.violin(adata, keys='percent_oxphos', groupby=groupby, rotation=90,size=0.1,palette="pastel",linewidth=0,ax=axs[3,1], show=False)
sc.pl.violin(adata, keys='percent_hemo', groupby=groupby, rotation=90,size=0.1,palette="pastel",linewidth=0,ax=axs[4,0], show=False)
sc.pl.violin(adata, keys='G2M.Score', groupby=groupby, rotation=90,size=0.1,palette="pastel",linewidth=0,ax=axs[4,1], show=False)

fig.tight_layout()
plt.savefig(f'{obj_path}{celltype}/{groupby}_scVI_L2_metadata.png')

In [326]:
adata.obs.groupby([groupby,'receptor_type']).size()

L2_leiden_scVI_1  receptor_type
0                 TCR              1283
1                 TCR               130
2                 TCR               827
3                 TCR              1562
4                 TCR              1147
5                 TCR               895
6                 TCR               871
7                 TCR              1024
8                 TCR               176
9                 TCR               592
10                TCR               832
11                TCR               426
12                TCR              1174
13                TCR               710
14                TCR              1638
15                TCR               761
16                TCR               338
17                TCR               590
18                TCR               458
19                TCR               983
20                TCR               443
dtype: int64

In [327]:
sc.tl.dendrogram(adata,groupby=groupby,use_rep='X_scVI')

In [ ]:
#cosg差异
cosg.cosg(adata, key_added=f'cosg_{groupby}', groupby=groupby,
          mu=10,n_genes_user=100,remove_lowly_expressed=True,
         )
df_tmp = pd.DataFrame(adata.uns[f'cosg_{groupby}']['names'])
df_tmp.to_csv(f"{obj_path}{celltype}/cosg_{groupby}.csv")
#cosg差异作图
df_tmp=pd.DataFrame(adata.uns[f'cosg_{groupby}']['names'][:8,]).T
df_tmp=df_tmp.reindex(adata.uns['dendrogram_'+groupby]['categories_ordered'])
marker_genes_list={idx: list(row.values) for idx, row in df_tmp.iterrows()}
marker_genes_list = {k: v for k, v in marker_genes_list.items() if not any(isinstance(x, float) for x in v)}
sc.pl.dotplot(adata, marker_genes_list,
             groupby=groupby,
             dendrogram=True,
             swap_axes=False,
             standard_scale='var',
              save=f'cosg_{groupby}',
             cmap='Spectral_r')

In [329]:
FeatureMatrix = pd.DataFrame(adata.uns[f'cosg_{groupby}']['names'])

In [ ]:
%%R -i FeatureMatrix,R_data_path,groupby
source('GetEnrich.R')
GetEnrich(FeatureMatrix,plot_term_number = 10,R_data_path=R_data_path,groupby=groupby,
          gmtfile="/localdisk/immune/Marker/msigdb_v2024.1.Hs_GMTs/h.all.v2024.1.Hs.symbols.gmt")
#"/localdisk/immune/Marker/Immunity_PBMC.gmt"
#"/localdisk/immune/Marker/sepsis_PBMC.gmt"
#"/localdisk/immune/Marker/msigdb_v2024.1.Hs_GMTs/c5.go.bp.v2024.1.Hs.symbols.gmt"
#"/localdisk/immune/Marker/msigdb_v2024.1.Hs_GMTs/c2.cp.kegg_legacy.v2024.1.Hs.symbols.gmt"

In [ ]:
fig, axs = plt.subplots(ncols=4, nrows=2, figsize=(16, 8))
sc.pl.umap(adata, color=groupby,legend_fontsize=4, legend_fontoutline=2,size=4,legend_loc='on data',ax=axs[0,0],show=False)
sc.pl.umap(adata, color='CSF3R',legend_fontsize=4, legend_fontoutline=2,size=4,legend_loc='on data',ax=axs[0,1],show=False)
sc.pl.umap(adata, color='CD14',legend_fontsize=4, legend_fontoutline=2,size=4,legend_loc='on data',ax=axs[0,2],show=False)
sc.pl.umap(adata, color='CD4',legend_fontsize=4, legend_fontoutline=2,size=4,legend_loc='on data',ax=axs[0,3],show=False)
#sc.pl.violin(adt,'CD45RA',groupby=groupby,show=False, ax=axs[0,3],size=0.5)
#TNFAIP3/CXCR4/DDIT4/FOSL2/IRS2/BTG1/ZFP36/PNRC1/CDKN1B/PLIN2/BHLHE40/PLAUR/ETS1/DUSP1
#SELL/TXNIP/IFITM2/B2M/HLA-C

sc.pl.violin(adata,'CD3D',groupby=groupby,show=False, ax=axs[1,0],size=0.5)
sc.pl.violin(adata,'KLRB1',groupby=groupby,show=False, ax=axs[1,1],size=0.5)
sc.pl.violin(adata,'TRAV1-2',groupby=groupby,show=False, ax=axs[1,2],size=0.5)
sc.pl.violin(adata,'SLC4A10',groupby=groupby,show=False, ax=axs[1,3],size=0.5)
plt.show()

In [ ]:
sc.pl.violin(adata,'',groupby=groupby,size=0.5)

In [349]:
cell_dict={
    'Doublet':['1',#mono + scDblFinder
               '6',#Neu + scDblFinder
               '4',#NK + scDblFinder
               '18',#CD4+ CD8+
               '8',#γδ T
              ],

    'MAIT':['0','3','5','11','12','17','20',#CD4 MAIT?
        '2','7','9','10','13','14','15','16','19'#CD8+ TRAV1-2+ KLRB1+ SLC4A10+
            ],

}

In [350]:
check_dict_duplicates(cell_dict)

'无重复'

In [351]:
adata.obs[groupby].value_counts()

L2_leiden_scVI_1
14    2312
3     1872
4     1560
7     1521
19    1484
12    1406
0     1386
6     1180
10    1144
5     1055
2     1054
9      986
13     967
15     875
17     634
8      607
18     563
20     548
16     538
11     473
1      149
Name: count, dtype: int64

In [352]:
# Generate new assignments
for i in cell_dict.keys():
    ind = pd.Series(adata.obs[groupby]).isin(cell_dict[i])
    adata.obs.loc[ind,'Celltype_L1_L2_Refine'] = i

In [353]:
(adata.obs['Celltype_L1_L2_Refine'].isna()).value_counts()

Celltype_L1_L2_Refine
False    22314
Name: count, dtype: int64

In [354]:
adata.obs['Celltype_L1_L2_Refine'].value_counts()

Celltype_L1_L2_Refine
MAIT       18255
Doublet     4059
Name: count, dtype: int64

In [355]:
adata.obs.loc[adata.obs['receptor_type_BCR']=='BCR','Celltype_L1_L2_Refine']="Doublet"

In [356]:
adata.obs['Celltype_L1_L2_Refine'].value_counts()

Celltype_L1_L2_Refine
MAIT       17983
Doublet     4331
Name: count, dtype: int64

In [357]:
adata = adata[adata.obs['Celltype_L1_L2_Refine'] != "Doublet",:]

In [358]:
adata.write(f"{obj_path}{celltype}/{celltype}_L2_Refine.h5ad",compression="gzip")

# 7. gdT Refine

In [359]:
celltype="gdT"

In [360]:
adata = sc.read_h5ad(f"{obj_path}{celltype}/{celltype}_after_cluster_scVI.h5ad")
R_data_path = f"{obj_path}{celltype}"
sc.settings.figdir=f"{obj_path}{celltype}"

In [361]:
sc.pp.normalize_total(adata, target_sum=1e4)
sc.pp.log1p(adata)

In [362]:
del adata.obs['Celltype_L1_L2_Refine']

In [363]:
pd.DataFrame(adata.obs[['L2_leiden_scVI_0.5', 'L2_leiden_scVI_1', 'L2_leiden_scVI_1.5', 'L2_leiden_scVI_2']]).to_csv(f'{R_data_path}/Level2_scVI_ForClustree.csv')
os.system(f'/home/liyanguo/anaconda3/envs/R/bin/Rscript ref_clustree.R {R_data_path} &')

0

In [ ]:
sc.pl.umap(adata, color=['receptor_type','receptor_type_BCR'],
           frameon=False,
           legend_fontsize=4, legend_fontoutline=2, 
           size=4,
           legend_loc='on data')

adata.obs['cluster_dummy'] = adata.obs[groupby] == adata.obs[groupby].cat.categories[5]
sc.pl.umap(adata, color='cluster_dummy',legend_fontsize=4, legend_fontoutline=2,size=4,legend_loc='on data')

In [387]:
groupby = "L2_leiden_scVI_1"

In [388]:
xlsx = pd.ExcelWriter(f"{obj_path}{celltype}/{groupby}_freq_table.xlsx")
for group in ['SampleID', 'DonorID','scDblFinder.class','Phase',
              'Immune_All_High', 'Immune_All_Low', 'Adult_Human_Blood', 'Adult_Human_Bone_marrow',
              'AIFI_L1', 'AIFI_L2', 'AIFI_L3', 'predicted.celltype.l1', 'predicted.celltype.l2',
              'predicted.celltype.l3',
              'receptor_subtype', #TCR
              'VJ_1_v_call', 'VJ_1_j_call', 
              'VDJ_1_v_call', 'VDJ_1_j_call',
              'receptor_type_BCR', #BCR
              'VJ_1_v_call_BCR','VJ_1_j_call_BCR', 
              'VDJ_1_v_call_BCR', 'VDJ_1_j_call_BCR']:
    freq_table_multi = adata.obs.groupby([groupby, group]).size()
    pd.DataFrame(freq_table_multi).to_excel(xlsx,sheet_name=group)
xlsx.close()

In [ ]:
#数据簇间信息比较
fig, axs = plt.subplots(5, 2, figsize=(18, 12),constrained_layout=True)
plt.subplots_adjust(hspace=1,wspace=1)
sc.pl.violin(adata, keys='nCount_RNA', groupby=groupby, rotation=90,size=0.1,palette="pastel",linewidth=0,ax=axs[0,0], show=False)
sc.pl.violin(adata, keys='nFeature_RNA', groupby=groupby, rotation=90,size=0.1,palette="pastel",linewidth=0,ax=axs[0,1], show=False)
sc.pl.violin(adata, keys='log10GenesPerUMI', groupby=groupby, rotation=90,size=0.1,palette="pastel",linewidth=0,ax=axs[1,0], show=False)
sc.pl.violin(adata, keys='pct_counts_in_top_20_genes', groupby=groupby, rotation=90,size=0.1,palette="pastel",linewidth=0,ax=axs[1,1], show=False)

sc.pl.violin(adata, keys='percent_apop', groupby=groupby, rotation=90,size=0.1,palette="pastel",linewidth=0,ax=axs[2,0], show=False)
sc.pl.violin(adata, keys='percent_ribo', groupby=groupby, rotation=90,size=0.1,palette="pastel",linewidth=0,ax=axs[2,1], show=False)
sc.pl.violin(adata, keys='percent_ieg', groupby=groupby, rotation=90,size=0.1,palette="pastel",linewidth=0,ax=axs[3,0], show=False)
sc.pl.violin(adata, keys='percent_oxphos', groupby=groupby, rotation=90,size=0.1,palette="pastel",linewidth=0,ax=axs[3,1], show=False)
sc.pl.violin(adata, keys='percent_hemo', groupby=groupby, rotation=90,size=0.1,palette="pastel",linewidth=0,ax=axs[4,0], show=False)
sc.pl.violin(adata, keys='G2M.Score', groupby=groupby, rotation=90,size=0.1,palette="pastel",linewidth=0,ax=axs[4,1], show=False)

fig.tight_layout()
plt.savefig(f'{obj_path}{celltype}/{groupby}_scVI_L2_metadata.png')

In [390]:
adata.obs.groupby([groupby,'receptor_type']).size()

L2_leiden_scVI_1  receptor_type
0                 TCR               21
1                 TCR              598
2                 TCR              287
3                 TCR              258
4                 TCR              354
5                 TCR               76
6                 TCR              260
7                 TCR              127
8                 TCR              126
9                 TCR               73
10                TCR              126
11                TCR              103
12                TCR              193
13                TCR              139
14                TCR              183
15                TCR              166
16                TCR              121
17                TCR               97
18                TCR               76
19                TCR              271
20                TCR               47
dtype: int64

In [391]:
sc.tl.dendrogram(adata,groupby=groupby,use_rep='X_scVI')

In [ ]:
#cosg差异
cosg.cosg(adata, key_added=f'cosg_{groupby}', groupby=groupby,
          mu=10,n_genes_user=100,remove_lowly_expressed=True,
         )
df_tmp = pd.DataFrame(adata.uns[f'cosg_{groupby}']['names'])
df_tmp.to_csv(f"{obj_path}{celltype}/cosg_{groupby}.csv")
#cosg差异作图
df_tmp=pd.DataFrame(adata.uns[f'cosg_{groupby}']['names'][:8,]).T
df_tmp=df_tmp.reindex(adata.uns['dendrogram_'+groupby]['categories_ordered'])
marker_genes_list={idx: list(row.values) for idx, row in df_tmp.iterrows()}
marker_genes_list = {k: v for k, v in marker_genes_list.items() if not any(isinstance(x, float) for x in v)}
sc.pl.dotplot(adata, marker_genes_list,
             groupby=groupby,
             dendrogram=True,
             swap_axes=False,
             standard_scale='var',
              save=f'cosg_{groupby}',
             cmap='Spectral_r')

In [393]:
FeatureMatrix = pd.DataFrame(adata.uns[f'cosg_{groupby}']['names'])

In [ ]:
%%R -i FeatureMatrix,R_data_path,groupby
source('GetEnrich.R')
GetEnrich(FeatureMatrix,plot_term_number = 10,R_data_path=R_data_path,groupby=groupby,
          gmtfile="/localdisk/immune/Marker/msigdb_v2024.1.Hs_GMTs/h.all.v2024.1.Hs.symbols.gmt")
#"/localdisk/immune/Marker/Immunity_PBMC.gmt"
#"/localdisk/immune/Marker/sepsis_PBMC.gmt"
#"/localdisk/immune/Marker/msigdb_v2024.1.Hs_GMTs/c5.go.bp.v2024.1.Hs.symbols.gmt"
#"/localdisk/immune/Marker/msigdb_v2024.1.Hs_GMTs/c2.cp.kegg_legacy.v2024.1.Hs.symbols.gmt"

In [ ]:
fig, axs = plt.subplots(ncols=4, nrows=2, figsize=(16, 8))
sc.pl.umap(adata, color=groupby,legend_fontsize=4, legend_fontoutline=2,size=4,legend_loc='on data',ax=axs[0,0],show=False)
sc.pl.umap(adata, color='TRDV1',legend_fontsize=4, legend_fontoutline=2,size=4,legend_loc='on data',ax=axs[0,1],show=False)
sc.pl.umap(adata, color='TIGIT',legend_fontsize=4, legend_fontoutline=2,size=4,legend_loc='on data',ax=axs[0,2],show=False)
sc.pl.umap(adata, color='KLRF1',legend_fontsize=4, legend_fontoutline=2,size=4,legend_loc='on data',ax=axs[0,3],show=False)

sc.pl.violin(adata,'CD8A',groupby=groupby,show=False, ax=axs[1,0],size=0.5)
sc.pl.violin(adata,'CD8B',groupby=groupby,show=False, ax=axs[1,1],size=0.5)
sc.pl.violin(adata,'CD3D',groupby=groupby,show=False, ax=axs[1,2],size=0.5)
sc.pl.violin(adata,'CD27',groupby=groupby,show=False, ax=axs[1,3],size=0.5)
plt.show()

In [ ]:
sc.pl.violin(adata,'TRDV1',groupby=groupby,size=0.5)

In [397]:
adata.obs[groupby].value_counts()

L2_leiden_scVI_1
13    1200
1     1043
2      993
8      984
4      917
12     914
5      804
9      795
17     676
19     675
10     670
14     646
16     638
6      560
11     547
15     541
18     527
7      516
3      475
20     401
0       54
Name: count, dtype: int64

In [400]:
cell_dict={
    'Doublet':['0',#monocyte
               '11',#Neu + scDblFinder
               '1',#CD4 γδ T
               '2','3','4','6','7','10','13','14','16'#NK CD3D+
              ],
    
    'γδ T':['5','8','9','12','15','17','18','19','20'],
}

#gdT 在血液中是CD45RA- CD27+/-
# #γδ T；https://www.nature.com/articles/s41467-018-04076-0；
# https://www.nature.com/articles/s41392-023-01653-8#Sec12
# Vδ2+ T cells were predominantly CD27+CD45RA−
# γδT细胞根据TCR的γ（包括2/3/4/5/8/9）和δ（包括1/2/3/5）链的表达，γδT细胞主要分为三个亚群：Vδ1T细胞，Vδ2T细胞和Vδ3T细胞。
# Vδ1T细胞主要存在于粘膜上皮细胞中，Vδ2T细胞主要分布在外周血中，Vδ3T细胞主要分布肝和肠。
# 基于CD27和CD45RA的表达差异Vδ2T细胞又分为CD45RA + CD27 +（幼稚），CD45RA-CD27 +（中间，无记忆效应功能），
# CD45RA-CD27-（记忆效应）和CD45RA + CD27-（终末分化）四群。
# 另外，γδT细胞也可分为多个功能亚群：产IFN-γ，产IL-17AγδT细胞和抗原呈递γδT细胞。

In [401]:
check_dict_duplicates(cell_dict)

'无重复'

In [402]:
# Generate new assignments
for i in cell_dict.keys():
    ind = pd.Series(adata.obs[groupby]).isin(cell_dict[i])
    adata.obs.loc[ind,'Celltype_L1_L2_Refine'] = i

In [403]:
adata.obs['Celltype_L1_L2_Refine'].value_counts()

Celltype_L1_L2_Refine
Doublet    8259
γδ T       6317
Name: count, dtype: int64

In [404]:
(adata.obs['Celltype_L1_L2_Refine'].isna()).value_counts()

Celltype_L1_L2_Refine
False    14576
Name: count, dtype: int64

In [405]:
adata.obs.loc[adata.obs['receptor_type_BCR']=='BCR','Celltype_L1_L2_Refine']="Doublet"
adata.obs.loc[adata.obs['receptor_type']=='TCR','Celltype_L1_L2_Refine']="Doublet"

In [406]:
adata.obs['Celltype_L1_L2_Refine'].value_counts()

Celltype_L1_L2_Refine
Doublet    9467
γδ T       5109
Name: count, dtype: int64

In [407]:
adata = adata[adata.obs['Celltype_L1_L2_Refine'] != "Doublet",:]

In [408]:
adata.obs['Celltype_L1_L2_Refine'].value_counts()

Celltype_L1_L2_Refine
γδ T    5109
Name: count, dtype: int64

In [409]:
adata.write(f"{obj_path}{celltype}/{celltype}_L2_Refine.h5ad",compression="gzip")

# 8. iNKT Refine

In [609]:
celltype="iNKT"

In [610]:
adata = sc.read_h5ad(f"{obj_path}{celltype}/{celltype}.h5ad")
R_data_path = f"{obj_path}{celltype}"
sc.settings.figdir=f"{obj_path}{celltype}"

In [611]:
sc.pp.normalize_total(adata, target_sum=1e4)
sc.pp.log1p(adata)

In [ ]:
sc.pl.umap(adata, color=['receptor_type','receptor_type_BCR'],
           frameon=False,
           legend_fontsize=4, legend_fontoutline=2, 
           size=4,
           legend_loc='on data')

In [613]:
xlsx = pd.ExcelWriter(f"{obj_path}{celltype}/freq_table.xlsx")
for group in ['SampleID', 'DonorID','scDblFinder.class','Phase',
              'Immune_All_High', 'Immune_All_Low', 'Adult_Human_Blood', 'Adult_Human_Bone_marrow',
              'AIFI_L1', 'AIFI_L2', 'AIFI_L3', 'predicted.celltype.l1', 'predicted.celltype.l2',
              'predicted.celltype.l3',
              'receptor_subtype', #TCR
              'VJ_1_v_call', 'VJ_1_j_call', 
              'VDJ_1_v_call', 'VDJ_1_j_call',
              'receptor_type_BCR', #BCR
              'VJ_1_v_call_BCR','VJ_1_j_call_BCR', 
              'VDJ_1_v_call_BCR', 'VDJ_1_j_call_BCR']:
    freq_table_multi = adata.obs.groupby([group]).size()
    pd.DataFrame(freq_table_multi).to_excel(xlsx,sheet_name=group)
xlsx.close()

In [614]:
adata.obs['Celltype_L1_L2_Refine'].value_counts()

Celltype_L1_L2_Refine
iNKT    44
Name: count, dtype: int64

In [615]:
adata.obs['Celltype_L1_L2_Refine']=adata.obs['Celltype_L1_L2_Refine'].cat.add_categories('Doublet')

In [616]:
adata.obs.loc[adata.obs['receptor_type_BCR']=='BCR','Celltype_L1_L2_Refine']="Doublet"

In [617]:
adata.obs['Celltype_L1_L2_Refine'].value_counts()

Celltype_L1_L2_Refine
iNKT       42
Doublet     2
Name: count, dtype: int64

In [618]:
adata = adata[adata.obs['Celltype_L1_L2_Refine'] != "Doublet",:]

In [619]:
adata.write(f"{obj_path}{celltype}/{celltype}_L2_Refine.h5ad",compression="gzip")

# 9. NK Cell Refine

In [39]:
celltype="NK"

In [40]:
adata = sc.read_h5ad(f"{obj_path}{celltype}/{celltype}_after_cluster_scVI.h5ad")
R_data_path = f"{obj_path}{celltype}"
sc.settings.figdir=f"{obj_path}{celltype}"

In [41]:
sc.pp.normalize_total(adata, target_sum=1e4)
sc.pp.log1p(adata)

In [42]:
del adata.obs['Celltype_L1_L2_Refine']

In [ ]:
sc.pl.umap(adata, color=['receptor_type','receptor_type_BCR'],
           frameon=False,
           legend_fontsize=4, legend_fontoutline=2, 
           size=4,
           legend_loc='on data')

In [44]:
pd.DataFrame(adata.obs[['L2_leiden_scVI_0.5', 'L2_leiden_scVI_1', 'L2_leiden_scVI_1.5', 'L2_leiden_scVI_2']]).to_csv(f'{R_data_path}/Level2_scVI_ForClustree.csv')
os.system(f'/home/liyanguo/anaconda3/envs/R/bin/Rscript ref_clustree.R {R_data_path} &')

0

In [45]:
groupby = "L2_leiden_scVI_1"

In [72]:
xlsx = pd.ExcelWriter(f"{obj_path}{celltype}/{groupby}_freq_table.xlsx")
for group in ['SampleID', 'DonorID','scDblFinder.class','Phase',
              'Immune_All_High', 'Immune_All_Low', 'Adult_Human_Blood', 'Adult_Human_Bone_marrow',
              'AIFI_L1', 'AIFI_L2', 'AIFI_L3', 'predicted.celltype.l1', 'predicted.celltype.l2',
              'predicted.celltype.l3',
              'receptor_subtype', #TCR
              'VJ_1_v_call', 'VJ_1_j_call', 
              'VDJ_1_v_call', 'VDJ_1_j_call',
              'receptor_type_BCR', #BCR
              'VJ_1_v_call_BCR','VJ_1_j_call_BCR', 
              'VDJ_1_v_call_BCR', 'VDJ_1_j_call_BCR']:
    freq_table_multi = adata.obs.groupby([groupby, group]).size()
    pd.DataFrame(freq_table_multi).to_excel(xlsx,sheet_name=group)
xlsx.close()

In [ ]:
#数据簇间信息比较
fig, axs = plt.subplots(5, 2, figsize=(18, 12),constrained_layout=True)
plt.subplots_adjust(hspace=1,wspace=1)
sc.pl.violin(adata, keys='nCount_RNA', groupby=groupby, rotation=90,size=0.1,palette="pastel",linewidth=0,ax=axs[0,0], show=False)
sc.pl.violin(adata, keys='nFeature_RNA', groupby=groupby, rotation=90,size=0.1,palette="pastel",linewidth=0,ax=axs[0,1], show=False)
sc.pl.violin(adata, keys='log10GenesPerUMI', groupby=groupby, rotation=90,size=0.1,palette="pastel",linewidth=0,ax=axs[1,0], show=False)
sc.pl.violin(adata, keys='pct_counts_in_top_20_genes', groupby=groupby, rotation=90,size=0.1,palette="pastel",linewidth=0,ax=axs[1,1], show=False)

sc.pl.violin(adata, keys='percent_apop', groupby=groupby, rotation=90,size=0.1,palette="pastel",linewidth=0,ax=axs[2,0], show=False)
sc.pl.violin(adata, keys='percent_ribo', groupby=groupby, rotation=90,size=0.1,palette="pastel",linewidth=0,ax=axs[2,1], show=False)
sc.pl.violin(adata, keys='percent_ieg', groupby=groupby, rotation=90,size=0.1,palette="pastel",linewidth=0,ax=axs[3,0], show=False)
sc.pl.violin(adata, keys='percent_oxphos', groupby=groupby, rotation=90,size=0.1,palette="pastel",linewidth=0,ax=axs[3,1], show=False)
sc.pl.violin(adata, keys='percent_hemo', groupby=groupby, rotation=90,size=0.1,palette="pastel",linewidth=0,ax=axs[4,0], show=False)
sc.pl.violin(adata, keys='G2M.Score', groupby=groupby, rotation=90,size=0.1,palette="pastel",linewidth=0,ax=axs[4,1], show=False)

fig.tight_layout()
plt.savefig(f'{obj_path}{celltype}/{groupby}_scVI_L2_metadata.png')

adata.obs['cluster_dummy'] = adata.obs[groupby] == adata.obs[groupby].cat.categories[10]
sc.pl.umap(adata, color='cluster_dummy',legend_fontsize=4, legend_fontoutline=2,size=4,legend_loc='on data')

In [48]:
sc.tl.dendrogram(adata,groupby=groupby,use_rep='X_scVI')#

In [ ]:
#cosg差异
cosg.cosg(adata, key_added=f'cosg_{groupby}', groupby=groupby,
          mu=10,n_genes_user=100,remove_lowly_expressed=True,
         )
df_tmp = pd.DataFrame(adata.uns[f'cosg_{groupby}']['names'])
df_tmp.to_csv(f"{obj_path}{celltype}/cosg_{groupby}.csv")
#cosg差异作图
df_tmp=pd.DataFrame(adata.uns[f'cosg_{groupby}']['names'][:8,]).T
df_tmp=df_tmp.reindex(adata.uns['dendrogram_'+groupby]['categories_ordered'])
marker_genes_list={idx: list(row.values) for idx, row in df_tmp.iterrows()}
marker_genes_list = {k: v for k, v in marker_genes_list.items() if not any(isinstance(x, float) for x in v)}
sc.pl.dotplot(adata, marker_genes_list,
             groupby=groupby,
             dendrogram=True,
             swap_axes=False,
             standard_scale='var',
             save=f'cosg_{groupby}',
             cmap='Spectral_r')

In [ ]:
signature= {
    'NK':['IRF8','JAK1','NKG7','PLEK','ZEB2','EOMES','GZMA','PRF1',],
    'Cytotoxicity':['GZMA', 'GZMB', 'GZMH', 'GZMM', 'GZMK', 'GNLY', 'PRF1','CTSW'],
    'Inflammatory':['CCL2', 'CCL3', 'CCL4', 'CCL5', 'CXCL10', 'CXCL9', 'IL1B', 'IL6', 'IL7', 'IL15' , 'IL18'],
    'Stress':['BAG3', 'CALU', 'DNAJB1', 'DUSP1', 'EGR1', 'FOS', 'FOSB', 'HIF1A', 'HSP90AA1', 'HSP90AB1',
                         'HSP90B1','HSPA1A', 'HSPA1B',  'HSPB1', 'HSPH1', 'IER2', 'JUN', 'JUNB', 'NFKBIA',#'HSPA6',
                         'NFKBIZ', 'RGS2', 'SLC2A3', 'SOCS3', 'UBC', 'ZFAND2A', 'ZFP36', 'ZFP36L1'],
    'HLA-dependent inhibitory receptors':['KIR3DL1', 'KIR3DL2', 'LILRB1','LAG3'],#'KIR2DL1', 'KIR2DL3', 
    'HLA-independent inhibitory receptors':['PDCD1', 'SIGLEC7', 'CD300A', 'CD96', 'IL1RAPL1', 'TIGIT', 'HAVCR2'],
    'HLA-dependent activating receptors':['KIR2DL4', 'CD160', 'KLRC2'],
    'HLA-independent activating receptors':['NCR3', 'NCR1',  'CRTAM','FCGR3A'],#'KLRK1',
}
sc.pl.dotplot(adata, signature,
             groupby=groupby,
             dendrogram=True,
             swap_axes=False,
             standard_scale='var',
             save=f'signature_{groupby}',
             cmap='Spectral_r')

In [ ]:
#NK基因打分
sc.tl.score_genes(adata,signature['NK'])
fig, axs = plt.subplots(ncols=2, nrows=1, figsize=(10, 6))
sc.pl.umap(adata, color='score',legend_fontsize=4, legend_fontoutline=2,size=4,legend_loc='on data',show=False, ax=axs[0])
sc.pl.violin(adata,'score',groupby=groupby,show=False, ax=axs[1],size=0)
plt.show()

In [ ]:
fig, axs = plt.subplots(ncols=4, nrows=2, figsize=(16, 8))
sc.pl.umap(adata, color=groupby,legend_fontsize=4, legend_fontoutline=2,size=4,legend_loc='on data',ax=axs[0,0],show=False)
sc.pl.umap(adata, color='CD3D',legend_fontsize=4, legend_fontoutline=2,size=4,legend_loc='on data',ax=axs[0,1],show=False)
sc.pl.umap(adata, color='CD7',legend_fontsize=4, legend_fontoutline=2,size=4,legend_loc='on data',ax=axs[0,2],show=False)
sc.pl.umap(adata, color='CD14',legend_fontsize=4, legend_fontoutline=2,size=4,legend_loc='on data',ax=axs[0,3],show=False)

sc.pl.violin(adata,'FCER1G',groupby=groupby,show=False, ax=axs[1,0],size=0)
sc.pl.violin(adata,'PRF1',groupby=groupby,show=False, ax=axs[1,1],size=0)
sc.pl.violin(adata,'HLA-DRA',groupby=groupby,show=False, ax=axs[1,2],size=0)
sc.pl.violin(adata,'FCGR3A',groupby=groupby,show=False, ax=axs[1,3],size=0)
plt.show()

In [ ]:
sc.pl.violin(adata,'CD3D',groupby=groupby,size=0)

In [54]:
FeatureMatrix = pd.DataFrame(adata.uns[f'cosg_{groupby}']['names'])

In [ ]:
%%R -i FeatureMatrix,R_data_path,groupby
source('GetEnrich.R')
GetEnrich(FeatureMatrix,plot_term_number = 10,R_data_path=R_data_path,groupby=groupby,
          gmtfile="/localdisk/immune/Marker/msigdb_v2024.1.Hs_GMTs/h.all.v2024.1.Hs.symbols.gmt")
#"/localdisk/immune/Marker/msigdb_v2024.1.Hs_GMTs/h.all.v2024.1.Hs.symbols.gmt"
#"/localdisk/immune/Marker/Immunity_PBMC.gmt"
#"/localdisk/immune/Marker/sepsis_PBMC.gmt"
#"/localdisk/immune/Marker/msigdb_v2024.1.Hs_GMTs/c5.go.bp.v2024.1.Hs.symbols.gmt"
#"/localdisk/immune/Marker/msigdb_v2024.1.Hs_GMTs/c2.cp.kegg_legacy.v2024.1.Hs.symbols.gmt"

In [74]:
cell_dict = {'Doublet':['4',#Mono
                        '7',#Neu + scDBlFinder
                        '13','14',#CD3D+ CD4+ IL7R+, scDBlFinder

                        
                        '10',#CD3D 有疑问
                        ],
             
             'NK cells':['6','9',#CD16low CD56+
                   '0','1','2','3','5','8','11','12',
                  ],
            }
#https://www.nature.com/articles/s41590-024-01883-0
#'Intermediate NK'

In [75]:
adata.obs[groupby].value_counts()

L2_leiden_scVI_1
12    17481
2     13918
11    12729
0     12268
14    12095
10    11342
1      9512
3      9105
7      8106
8      7146
6      6447
13     3027
5      1381
4      1313
9       879
Name: count, dtype: int64

In [76]:
check_dict_duplicates(cell_dict)

'无重复'

In [77]:
# Generate new assignments
for i in cell_dict.keys():
    ind = pd.Series(adata.obs[groupby]).isin(cell_dict[i])
    adata.obs.loc[ind,'Celltype_L1_L2_Refine'] = i

In [78]:
adata.obs['Celltype_L1_L2_Refine'].value_counts()

Celltype_L1_L2_Refine
NK         90866
Doublet    35883
Name: count, dtype: int64

In [79]:
(adata.obs['Celltype_L1_L2_Refine'].isna()).value_counts()

Celltype_L1_L2_Refine
False    126749
Name: count, dtype: int64

In [80]:
adata.obs.loc[adata.obs['receptor_type_BCR']=='BCR','Celltype_L1_L2_Refine']="Doublet"
adata.obs.loc[adata.obs['receptor_type']=='TCR','Celltype_L1_L2_Refine']="Doublet"

In [81]:
adata.obs['Celltype_L1_L2_Refine'].value_counts()

Celltype_L1_L2_Refine
NK         79020
Doublet    47729
Name: count, dtype: int64

In [82]:
adata = adata[adata.obs['Celltype_L1_L2_Refine'] != "Doublet",:]

In [83]:
adata.write(f"{obj_path}{celltype}/{celltype}_L2_Refine.h5ad",compression="gzip")

# 10. Monocyte Refine

In [186]:
celltype="Monocyte"

In [187]:
adata = sc.read_h5ad(f"{obj_path}{celltype}/{celltype}_after_cluster_scVI.h5ad")
R_data_path = f"{obj_path}{celltype}"
sc.settings.figdir=f"{obj_path}{celltype}"

In [188]:
sc.pp.normalize_total(adata, target_sum=1e4)
sc.pp.log1p(adata)

In [189]:
del adata.obs['Celltype_L1_L2_Refine']

In [ ]:
sc.pl.umap(adata, color=['receptor_type','receptor_type_BCR'],
           frameon=False,
           legend_fontsize=4, legend_fontoutline=2, 
           size=4,
           legend_loc='on data')

In [191]:
pd.DataFrame(adata.obs[['L2_leiden_scVI_0.5', 'L2_leiden_scVI_1', 'L2_leiden_scVI_1.5', 'L2_leiden_scVI_2']]).to_csv(f'{R_data_path}/Level2_scVI_ForClustree.csv')
os.system(f'/home/liyanguo/anaconda3/envs/R/bin/Rscript ref_clustree.R {R_data_path} &')

0

In [192]:
groupby = "L2_leiden_scVI_1"

In [193]:
xlsx = pd.ExcelWriter(f"{obj_path}{celltype}/{groupby}_freq_table.xlsx")
for group in ['SampleID', 'DonorID','scDblFinder.class','Phase',
              'Immune_All_High', 'Immune_All_Low', 'Adult_Human_Blood', 'Adult_Human_Bone_marrow',
              'AIFI_L1', 'AIFI_L2', 'AIFI_L3', 'predicted.celltype.l1', 'predicted.celltype.l2',
              'predicted.celltype.l3',
              'receptor_subtype', #TCR
              'VJ_1_v_call', 'VJ_1_j_call', 
              'VDJ_1_v_call', 'VDJ_1_j_call',
              'receptor_type_BCR', #BCR
              'VJ_1_v_call_BCR','VJ_1_j_call_BCR', 
              'VDJ_1_v_call_BCR', 'VDJ_1_j_call_BCR']:
    freq_table_multi = adata.obs.groupby([groupby, group]).size()
    pd.DataFrame(freq_table_multi).to_excel(xlsx,sheet_name=group)
xlsx.close()

Loading required package: ggraph
Loading required package: ggplot2


In [ ]:
#数据簇间信息比较
fig, axs = plt.subplots(5, 2, figsize=(18, 12),constrained_layout=True)
plt.subplots_adjust(hspace=1,wspace=1)
sc.pl.violin(adata, keys='nCount_RNA', groupby=groupby, rotation=90,size=0.1,palette="pastel",linewidth=0,ax=axs[0,0], show=False)
sc.pl.violin(adata, keys='nFeature_RNA', groupby=groupby, rotation=90,size=0.1,palette="pastel",linewidth=0,ax=axs[0,1], show=False)
sc.pl.violin(adata, keys='log10GenesPerUMI', groupby=groupby, rotation=90,size=0.1,palette="pastel",linewidth=0,ax=axs[1,0], show=False)
sc.pl.violin(adata, keys='pct_counts_in_top_20_genes', groupby=groupby, rotation=90,size=0.1,palette="pastel",linewidth=0,ax=axs[1,1], show=False)

sc.pl.violin(adata, keys='percent_apop', groupby=groupby, rotation=90,size=0.1,palette="pastel",linewidth=0,ax=axs[2,0], show=False)
sc.pl.violin(adata, keys='percent_ribo', groupby=groupby, rotation=90,size=0.1,palette="pastel",linewidth=0,ax=axs[2,1], show=False)
sc.pl.violin(adata, keys='percent_ieg', groupby=groupby, rotation=90,size=0.1,palette="pastel",linewidth=0,ax=axs[3,0], show=False)
sc.pl.violin(adata, keys='percent_oxphos', groupby=groupby, rotation=90,size=0.1,palette="pastel",linewidth=0,ax=axs[3,1], show=False)
sc.pl.violin(adata, keys='percent_hemo', groupby=groupby, rotation=90,size=0.1,palette="pastel",linewidth=0,ax=axs[4,0], show=False)
sc.pl.violin(adata, keys='G2M.Score', groupby=groupby, rotation=90,size=0.1,palette="pastel",linewidth=0,ax=axs[4,1], show=False)

fig.tight_layout()
plt.savefig(f'{obj_path}{celltype}/{groupby}_scVI_L2_metadata.png')

adata.obs['cluster_dummy'] = adata.obs[groupby] == adata.obs[groupby].cat.categories[8]
sc.pl.umap(adata, color='cluster_dummy',legend_fontsize=4, legend_fontoutline=2,size=4,legend_loc='on data')

In [195]:
sc.tl.dendrogram(adata,groupby=groupby,use_rep='X_scVI')#

In [ ]:
#cosg差异
cosg.cosg(adata, key_added=f'cosg_{groupby}', groupby=groupby,
          mu=10,n_genes_user=100,remove_lowly_expressed=True,
         )
df_tmp = pd.DataFrame(adata.uns[f'cosg_{groupby}']['names'])
df_tmp.to_csv(f"{obj_path}{celltype}/cosg_{groupby}.csv")
#cosg差异作图
df_tmp=pd.DataFrame(adata.uns[f'cosg_{groupby}']['names'][:8,]).T
df_tmp=df_tmp.reindex(adata.uns['dendrogram_'+groupby]['categories_ordered'])
marker_genes_list={idx: list(row.values) for idx, row in df_tmp.iterrows()}
marker_genes_list = {k: v for k, v in marker_genes_list.items() if not any(isinstance(x, float) for x in v)}
sc.pl.dotplot(adata, marker_genes_list,
             groupby=groupby,
             dendrogram=True,
             swap_axes=False,
             standard_scale='var',
             save=f'cosg_{groupby}',
             cmap='Spectral_r')

In [197]:
FeatureMatrix = pd.DataFrame(adata.uns[f'cosg_{groupby}']['names'])

In [ ]:
%%R -i FeatureMatrix,R_data_path,groupby
source('GetEnrich.R')
GetEnrich(FeatureMatrix,plot_term_number = 10,R_data_path=R_data_path,groupby=groupby,
          gmtfile="/localdisk/immune/Marker/msigdb_v2024.1.Hs_GMTs/h.all.v2024.1.Hs.symbols.gmt")
#"/localdisk/immune/Marker/msigdb_v2024.1.Hs_GMTs/h.all.v2024.1.Hs.symbols.gmt"
#"/localdisk/immune/Marker/Immunity_PBMC.gmt"
#"/localdisk/immune/Marker/sepsis_PBMC.gmt"
#"/localdisk/immune/Marker/msigdb_v2024.1.Hs_GMTs/c5.go.bp.v2024.1.Hs.symbols.gmt"
#"/localdisk/immune/Marker/msigdb_v2024.1.Hs_GMTs/c2.cp.kegg_legacy.v2024.1.Hs.symbols.gmt"

In [ ]:
fig, axs = plt.subplots(ncols=4, nrows=2, figsize=(16, 8))
sc.pl.umap(adata, color=groupby,legend_fontsize=4, legend_fontoutline=2,size=4,legend_loc='on data',ax=axs[0,0],show=False)
sc.pl.umap(adata, color='CD14',legend_fontsize=4, legend_fontoutline=2,size=4,legend_loc='on data',ax=axs[0,1],show=False)
sc.pl.umap(adata, color='CSF3R',legend_fontsize=4, legend_fontoutline=2,size=4,legend_loc='on data',ax=axs[0,2],show=False)
sc.pl.umap(adata, color='CD3D',legend_fontsize=4, legend_fontoutline=2,size=4,legend_loc='on data',ax=axs[0,3],show=False)

sc.pl.violin(adata,'FCGR3B',groupby=groupby,show=False, ax=axs[1,0],size=0)
sc.pl.violin(adata,'FCGR3A',groupby=groupby,show=False, ax=axs[1,1],size=0)
sc.pl.violin(adata,'HLA-DRA',groupby=groupby,show=False, ax=axs[1,2],size=0)
sc.pl.violin(adata,'CDKN1C',groupby=groupby,show=False, ax=axs[1,3],size=0)
plt.show()

In [ ]:
sc.pl.violin(adata,'ISG15',groupby=groupby,size=0)

In [211]:
cell_dict = {'Doublet':['16',#Neu
                        '8','10','13',#TNK + scDblFinder.class
                        '0',#Mono 低质量的细胞
                       ],
             
             'Classical monocytes':['1','2','3','4','5','9','12',],
             
             'NDNs':['6','7','11',],
             
             "Non-classical monocytes":['14','15'],
            }

In [212]:
adata.obs[groupby].value_counts()

L2_leiden_scVI_1
14    16968
11    10555
5     10531
0     10310
1      9594
7      8223
9      7347
3      7018
12     5574
2      5561
6      4862
4      4098
10     2897
8      1480
13     1398
16     1036
15      702
Name: count, dtype: int64

In [213]:
check_dict_duplicates(cell_dict)

'无重复'

In [214]:
# Generate new assignments
for i in cell_dict.keys():
    ind = pd.Series(adata.obs[groupby]).isin(cell_dict[i])
    adata.obs.loc[ind,'Celltype_L1_L2_Refine'] = i

In [215]:
(adata.obs['Celltype_L1_L2_Refine'].isna()).value_counts()

Celltype_L1_L2_Refine
False    108154
Name: count, dtype: int64

In [216]:
adata.obs.loc[adata.obs['receptor_type']=='TCR','Celltype_L1_L2_Refine']="Doublet"
adata.obs.loc[adata.obs['receptor_type_BCR']=='BCR','Celltype_L1_L2_Refine']="Doublet"

In [217]:
adata.obs['Celltype_L1_L2_Refine'].value_counts()

Celltype_L1_L2_Refine
Classical monocyte        44552
Doublet                   26253
CEACAM8- Neutrophil       21697
Non-classical monocyte    15652
Name: count, dtype: int64

In [218]:
adata = adata[adata.obs['Celltype_L1_L2_Refine'] != "Doublet",:]

In [219]:
adata.write(f"{obj_path}{celltype}/{celltype}_L2_Refine.h5ad",compression="gzip")

# 11. CEACAM8_Pos_Neutrophil (Low-density neutrophils) Refine

In [432]:
celltype="CEACAM8_Pos_Neutrophil"

In [433]:
adata = sc.read_h5ad(f"{obj_path}{celltype}/{celltype}_after_cluster_scVI.h5ad")
R_data_path = f"{obj_path}{celltype}"
sc.settings.figdir=f"{obj_path}{celltype}"

In [434]:
sc.pp.normalize_total(adata, target_sum=1e4)
sc.pp.log1p(adata)

In [435]:
del adata.obs['Celltype_L1_L2_Refine']

In [ ]:
sc.pl.umap(adata, color=['receptor_type','receptor_type_BCR'],
           frameon=False,
           legend_fontsize=4, legend_fontoutline=2, 
           size=4,
           legend_loc='on data')

In [437]:
pd.DataFrame(adata.obs[['L2_leiden_scVI_0.5', 'L2_leiden_scVI_1', 'L2_leiden_scVI_1.5', 'L2_leiden_scVI_2']]).to_csv(f'{R_data_path}/Level2_scVI_ForClustree.csv')
os.system(f'/home/liyanguo/anaconda3/envs/R/bin/Rscript ref_clustree.R {R_data_path} &')

0

Loading required package: ggraph
Loading required package: ggplot2


In [438]:
groupby = "L2_leiden_scVI_0.5"

In [439]:
xlsx = pd.ExcelWriter(f"{obj_path}{celltype}/{groupby}_freq_table.xlsx")
for group in ['SampleID', 'DonorID','scDblFinder.class','Phase',
              'Immune_All_High', 'Immune_All_Low', 'Adult_Human_Blood', 'Adult_Human_Bone_marrow',
              'AIFI_L1', 'AIFI_L2', 'AIFI_L3', 'predicted.celltype.l1', 'predicted.celltype.l2',
              'predicted.celltype.l3',
              'receptor_subtype', #TCR
              'VJ_1_v_call', 'VJ_1_j_call', 
              'VDJ_1_v_call', 'VDJ_1_j_call',
              'receptor_type_BCR', #BCR
              'VJ_1_v_call_BCR','VJ_1_j_call_BCR', 
              'VDJ_1_v_call_BCR', 'VDJ_1_j_call_BCR']:
    freq_table_multi = adata.obs.groupby([groupby, group]).size()
    pd.DataFrame(freq_table_multi).to_excel(xlsx,sheet_name=group)
xlsx.close()

In [ ]:
#数据簇间信息比较
fig, axs = plt.subplots(5, 2, figsize=(18, 12),constrained_layout=True)
plt.subplots_adjust(hspace=1,wspace=1)
sc.pl.violin(adata, keys='nCount_RNA', groupby=groupby, rotation=90,size=0.1,palette="pastel",linewidth=0,ax=axs[0,0], show=False)
sc.pl.violin(adata, keys='nFeature_RNA', groupby=groupby, rotation=90,size=0.1,palette="pastel",linewidth=0,ax=axs[0,1], show=False)
sc.pl.violin(adata, keys='log10GenesPerUMI', groupby=groupby, rotation=90,size=0.1,palette="pastel",linewidth=0,ax=axs[1,0], show=False)
sc.pl.violin(adata, keys='pct_counts_in_top_20_genes', groupby=groupby, rotation=90,size=0.1,palette="pastel",linewidth=0,ax=axs[1,1], show=False)

sc.pl.violin(adata, keys='percent_apop', groupby=groupby, rotation=90,size=0.1,palette="pastel",linewidth=0,ax=axs[2,0], show=False)
sc.pl.violin(adata, keys='percent_ribo', groupby=groupby, rotation=90,size=0.1,palette="pastel",linewidth=0,ax=axs[2,1], show=False)
sc.pl.violin(adata, keys='percent_ieg', groupby=groupby, rotation=90,size=0.1,palette="pastel",linewidth=0,ax=axs[3,0], show=False)
sc.pl.violin(adata, keys='percent_oxphos', groupby=groupby, rotation=90,size=0.1,palette="pastel",linewidth=0,ax=axs[3,1], show=False)
sc.pl.violin(adata, keys='percent_hemo', groupby=groupby, rotation=90,size=0.1,palette="pastel",linewidth=0,ax=axs[4,0], show=False)
sc.pl.violin(adata, keys='G2M.Score', groupby=groupby, rotation=90,size=0.1,palette="pastel",linewidth=0,ax=axs[4,1], show=False)

fig.tight_layout()
plt.savefig(f'{obj_path}{celltype}/{groupby}_scVI_L2_metadata.png')

adata.obs['cluster_dummy'] = adata.obs[groupby] == adata.obs[groupby].cat.categories[11]
sc.pl.umap(adata, color='cluster_dummy',legend_fontsize=4, legend_fontoutline=2,size=4,legend_loc='on data')

In [441]:
sc.tl.dendrogram(adata,groupby=groupby,use_rep='X_scVI')#

In [ ]:
#cosg差异
cosg.cosg(adata, key_added=f'cosg_{groupby}', groupby=groupby,
          mu=10,n_genes_user=100,remove_lowly_expressed=True,
         )
df_tmp = pd.DataFrame(adata.uns[f'cosg_{groupby}']['names'])
df_tmp.to_csv(f"{obj_path}{celltype}/cosg_{groupby}.csv")
#cosg差异作图
df_tmp=pd.DataFrame(adata.uns[f'cosg_{groupby}']['names'][:8,]).T
df_tmp=df_tmp.reindex(adata.uns['dendrogram_'+groupby]['categories_ordered'])
marker_genes_list={idx: list(row.values) for idx, row in df_tmp.iterrows()}
marker_genes_list = {k: v for k, v in marker_genes_list.items() if not any(isinstance(x, float) for x in v)}
sc.pl.dotplot(adata, marker_genes_list,
             groupby=groupby,
             dendrogram=True,
             swap_axes=False,
             standard_scale='var',
             save=f'cosg_{groupby}',
             cmap='Spectral_r')

In [443]:
FeatureMatrix = pd.DataFrame(adata.uns[f'cosg_{groupby}']['names'])

In [ ]:
%%R -i FeatureMatrix,R_data_path,groupby
source('GetEnrich.R')
GetEnrich(FeatureMatrix,plot_term_number = 10,R_data_path=R_data_path,groupby=groupby,
          gmtfile="/localdisk/immune/Marker/msigdb_v2024.1.Hs_GMTs/h.all.v2024.1.Hs.symbols.gmt")
#"/localdisk/immune/Marker/msigdb_v2024.1.Hs_GMTs/h.all.v2024.1.Hs.symbols.gmt"
#"/localdisk/immune/Marker/Immunity_PBMC.gmt"
#"/localdisk/immune/Marker/sepsis_PBMC.gmt"
#"/localdisk/immune/Marker/msigdb_v2024.1.Hs_GMTs/c5.go.bp.v2024.1.Hs.symbols.gmt"
#"/localdisk/immune/Marker/msigdb_v2024.1.Hs_GMTs/c2.cp.kegg_legacy.v2024.1.Hs.symbols.gmt"

In [ ]:
fig, axs = plt.subplots(ncols=4, nrows=2, figsize=(16, 8))
sc.pl.umap(adata, color=groupby,legend_fontsize=4, legend_fontoutline=2,size=4,legend_loc='on data',ax=axs[0,0],show=False)
sc.pl.umap(adata, color='MKI67',legend_fontsize=4, legend_fontoutline=2,size=4,legend_loc='on data',ax=axs[0,1],show=False)
sc.pl.umap(adata, color='MPO',legend_fontsize=4, legend_fontoutline=2,size=4,legend_loc='on data',ax=axs[0,2],show=False)
sc.pl.umap(adata, color='CD177',legend_fontsize=4, legend_fontoutline=2,size=4,legend_loc='on data',ax=axs[0,3],show=False)

sc.pl.violin(adata,'MKI67',groupby=groupby,show=False, ax=axs[1,0],size=0)
sc.pl.violin(adata,'CEACAM8',groupby=groupby,show=False, ax=axs[1,1],size=0)
sc.pl.violin(adata,'MME',groupby=groupby,show=False, ax=axs[1,2],size=0)
sc.pl.violin(adata,'ITGAM',groupby=groupby,show=False, ax=axs[1,3],size=0)
plt.show()

In [ ]:
sc.pl.violin(adata,'',groupby=groupby,size=0)

In [465]:
#都属于Immature Neutrophil，CD10- CD24+

cell_dict = {'Doublet':['10',#CEACAM8- Neutrophil MME+
                        '11',#Mono
                        '1','2','8',],#TNK scDblFinder
             'LDNs':['0','3','4','5','6','7',
                                   ],#
             'CD4+ T':['9'],
            }

#MPO,NE=ELANE,PAD4=PADI4,Gasdermin D=GSDMD,与中性粒NETs形成相关。

#maturation: ITGAM (CD11b), FCGR3A (CD16), and MME (CD10). https://www.nature.com/articles/s41590-023-01490-5

# https://www.biocompare.com/Editorial-Articles/577944-A-Guide-to-Neutrophil-Markers/
#  CD62Llow, CXCR2low, CXCR4hi, CD11bhi;
# upregulation of CXCR4 marks neutrophils for senescence;leading them back to the bone marrow for turnover.

#Degranulation, determined as an increase of the median fluorescent intensities (MFIs)
# of CD11b and CD66b;https://www.nature.com/articles/s41598-022-07455-2

#For mature, circulating neutrophils,
# a phenotype of CD16hi, CXCR2hi, CXCR4low, and CD62Lhi has been reported.

#Neutrophil profiling illuminates anti-tumor antigen-presenting potency

#黄来源：CD101区分成熟与否,但不适合

In [466]:
adata.obs[groupby].value_counts()

L2_leiden_scVI_0.5
2     488
9     437
3     378
6     356
0     315
4     304
7     302
10    273
8     217
5     171
1     104
11     53
Name: count, dtype: int64

In [467]:
check_dict_duplicates(cell_dict)

'无重复'

In [468]:
# Generate new assignments
for i in cell_dict.keys():
    ind = pd.Series(adata.obs[groupby]).isin(cell_dict[i])
    adata.obs.loc[ind,'Celltype_L1_L2_Refine'] = i

In [469]:
(adata.obs['Celltype_L1_L2_Refine'].isna()).value_counts()

Celltype_L1_L2_Refine
False    3398
Name: count, dtype: int64

In [470]:
adata.obs.loc[adata.obs['receptor_type']=='TCR','Celltype_L1_L2_Refine']="Doublet"
adata.obs.loc[adata.obs['receptor_type_BCR']=='BCR','Celltype_L1_L2_Refine']="Doublet"

In [471]:
adata.obs['Celltype_L1_L2_Refine'].value_counts()

Celltype_L1_L2_Refine
Doublet                1652
CEACAM8+ Neutrophil    1645
CD4+ T                  101
Name: count, dtype: int64

In [472]:
adata = adata[adata.obs['Celltype_L1_L2_Refine'] != "Doublet",:]

In [473]:
adata.write(f"{obj_path}{celltype}/{celltype}_L2_Refine.h5ad",compression="gzip")

# 12. CEACAM8_Neg_Neutrophil (Normal-density neutrophils)  Refine

In [133]:
celltype="CEACAM8_Neg_Neutrophil"

In [134]:
adata = sc.read_h5ad(f"{obj_path}{celltype}/{celltype}_after_cluster_scVI.h5ad")
R_data_path = f"{obj_path}{celltype}"
sc.settings.figdir=f"{obj_path}{celltype}"

In [135]:
sc.pp.normalize_total(adata, target_sum=1e4)
sc.pp.log1p(adata)

In [136]:
del adata.obs['Celltype_L1_L2_Refine']

In [ ]:
sc.pl.umap(adata, color=['receptor_type','receptor_type_BCR'],
           frameon=False,
           legend_fontsize=4, legend_fontoutline=2, 
           size=4,
           legend_loc='on data')

In [138]:
pd.DataFrame(adata.obs[['L2_leiden_scVI_0.5', 'L2_leiden_scVI_1', 'L2_leiden_scVI_1.5', 'L2_leiden_scVI_2']]).to_csv(f'{R_data_path}/Level2_scVI_ForClustree.csv')
os.system(f'/home/liyanguo/anaconda3/envs/R/bin/Rscript ref_clustree.R {R_data_path} &')

0

In [152]:
groupby = "L2_leiden_scVI_0.5"

In [140]:
xlsx = pd.ExcelWriter(f"{obj_path}{celltype}/{groupby}_freq_table.xlsx")
for group in ['SampleID', 'DonorID','scDblFinder.class','Phase',
              'Immune_All_High', 'Immune_All_Low', 'Adult_Human_Blood', 'Adult_Human_Bone_marrow',
              'AIFI_L1', 'AIFI_L2', 'AIFI_L3', 'predicted.celltype.l1', 'predicted.celltype.l2',
              'predicted.celltype.l3',
              'receptor_subtype', #TCR
              'VJ_1_v_call', 'VJ_1_j_call', 
              'VDJ_1_v_call', 'VDJ_1_j_call',
              'receptor_type_BCR', #BCR
              'VJ_1_v_call_BCR','VJ_1_j_call_BCR', 
              'VDJ_1_v_call_BCR', 'VDJ_1_j_call_BCR']:
    freq_table_multi = adata.obs.groupby([groupby, group]).size()
    pd.DataFrame(freq_table_multi).to_excel(xlsx,sheet_name=group)
xlsx.close()

Loading required package: ggraph
Loading required package: ggplot2


In [ ]:
#数据簇间信息比较
fig, axs = plt.subplots(5, 2, figsize=(18, 12),constrained_layout=True)
plt.subplots_adjust(hspace=1,wspace=1)
sc.pl.violin(adata, keys='nCount_RNA', groupby=groupby, rotation=90,size=0.1,palette="pastel",linewidth=0,ax=axs[0,0], show=False)
sc.pl.violin(adata, keys='nFeature_RNA', groupby=groupby, rotation=90,size=0.1,palette="pastel",linewidth=0,ax=axs[0,1], show=False)
sc.pl.violin(adata, keys='log10GenesPerUMI', groupby=groupby, rotation=90,size=0.1,palette="pastel",linewidth=0,ax=axs[1,0], show=False)
sc.pl.violin(adata, keys='pct_counts_in_top_20_genes', groupby=groupby, rotation=90,size=0.1,palette="pastel",linewidth=0,ax=axs[1,1], show=False)

sc.pl.violin(adata, keys='percent_apop', groupby=groupby, rotation=90,size=0.1,palette="pastel",linewidth=0,ax=axs[2,0], show=False)
sc.pl.violin(adata, keys='percent_ribo', groupby=groupby, rotation=90,size=0.1,palette="pastel",linewidth=0,ax=axs[2,1], show=False)
sc.pl.violin(adata, keys='percent_ieg', groupby=groupby, rotation=90,size=0.1,palette="pastel",linewidth=0,ax=axs[3,0], show=False)
sc.pl.violin(adata, keys='percent_oxphos', groupby=groupby, rotation=90,size=0.1,palette="pastel",linewidth=0,ax=axs[3,1], show=False)
sc.pl.violin(adata, keys='percent_hemo', groupby=groupby, rotation=90,size=0.1,palette="pastel",linewidth=0,ax=axs[4,0], show=False)
sc.pl.violin(adata, keys='G2M.Score', groupby=groupby, rotation=90,size=0.1,palette="pastel",linewidth=0,ax=axs[4,1], show=False)

fig.tight_layout()
plt.savefig(f'{obj_path}{celltype}/{groupby}_scVI_L2_metadata.png')

adata.obs['cluster_dummy'] = adata.obs[groupby] == adata.obs[groupby].cat.categories[17]
sc.pl.umap(adata, color='cluster_dummy',legend_fontsize=4, legend_fontoutline=2,size=4,legend_loc='on data')

In [142]:
sc.tl.dendrogram(adata,groupby=groupby,use_rep='X_scVI')#

In [ ]:
#cosg差异
cosg.cosg(adata, key_added=f'cosg_{groupby}', groupby=groupby,
          mu=10,n_genes_user=100,remove_lowly_expressed=True,
         )
df_tmp = pd.DataFrame(adata.uns[f'cosg_{groupby}']['names'])
df_tmp.to_csv(f"{obj_path}{celltype}/cosg_{groupby}.csv")
#cosg差异作图
df_tmp=pd.DataFrame(adata.uns[f'cosg_{groupby}']['names'][:8,]).T
df_tmp=df_tmp.reindex(adata.uns['dendrogram_'+groupby]['categories_ordered'])
marker_genes_list={idx: list(row.values) for idx, row in df_tmp.iterrows()}
marker_genes_list = {k: v for k, v in marker_genes_list.items() if not any(isinstance(x, float) for x in v)}
sc.pl.dotplot(adata, marker_genes_list,
             groupby=groupby,
             dendrogram=True,
             swap_axes=False,
             standard_scale='var',
             save=f'cosg_{groupby}',
             cmap='Spectral_r')

In [ ]:
fig, axs = plt.subplots(ncols=4, nrows=2, figsize=(16, 8))
sc.pl.umap(adata, color=groupby,legend_fontsize=4, legend_fontoutline=2,size=4,legend_loc='on data',ax=axs[0,0],show=False)
sc.pl.umap(adata, color='CD74',legend_fontsize=4, legend_fontoutline=2,size=4,legend_loc='on data',ax=axs[0,1],show=False)
sc.pl.umap(adata, color='MME',legend_fontsize=4, legend_fontoutline=2,size=4,legend_loc='on data',ax=axs[0,2],show=False)
sc.pl.umap(adata, color='CD2',legend_fontsize=4, legend_fontoutline=2,size=4,legend_loc='on data',ax=axs[0,3],show=False)

sc.pl.violin(adata,'FCER1G',groupby=groupby,show=False, ax=axs[1,0],size=0)
sc.pl.violin(adata,'CD3D',groupby=groupby,show=False, ax=axs[1,1],size=0)
sc.pl.violin(adata,'HLA-DRA',groupby=groupby,show=False, ax=axs[1,2],size=0)
sc.pl.violin(adata,'PTPRC',groupby=groupby,show=False, ax=axs[1,3],size=0)
plt.show()

In [ ]:
sc.pl.violin(adata,'CXCL8',groupby=groupby,size=0)

In [ ]:
sc.pl.violin(adata,'ADM',groupby=groupby,size=0)

In [147]:
FeatureMatrix = pd.DataFrame(adata.uns[f'cosg_{groupby}']['names'])

In [ ]:
%%R -i FeatureMatrix,R_data_path,groupby
source('GetEnrich.R')
GetEnrich(FeatureMatrix,plot_term_number = 10,R_data_path=R_data_path,groupby=groupby,
          gmtfile="/localdisk/immune/Marker/msigdb_v2024.1.Hs_GMTs/h.all.v2024.1.Hs.symbols.gmt")
#"/localdisk/immune/Marker/msigdb_v2024.1.Hs_GMTs/h.all.v2024.1.Hs.symbols.gmt"
#"/localdisk/immune/Marker/Immunity_PBMC.gmt"
#"/localdisk/immune/Marker/sepsis_PBMC.gmt"
#"/localdisk/immune/Marker/msigdb_v2024.1.Hs_GMTs/c5.go.bp.v2024.1.Hs.symbols.gmt"
#"/localdisk/immune/Marker/msigdb_v2024.1.Hs_GMTs/c2.cp.kegg_legacy.v2024.1.Hs.symbols.gmt"

In [153]:
cell_dict = {'Doublet':['5',
                       '6',
                       ],#
             'NDNs':['0','1','2',
                                    '3',#有混杂，但是增加分辨率，效果仍不佳
                                    '4',],
            }

#CXCR2/CXCL8促进出骨髓，CXCR4抑制出骨髓
#C5AR1 FPR1 ADM

#maturation: ITGAM (CD11b), FCGR3A (CD16), and MME (CD10). https://www.nature.com/articles/s41590-023-01490-5

# https://www.biocompare.com/Editorial-Articles/577944-A-Guide-to-Neutrophil-Markers/
#  CD62Llow, CXCR2low, CXCR4hi, CD11bhi;upregulation of CXCR4 marks neutrophils for senescence;leading them back to the bone marrow for turnover.

#Degranulation, determined as an increase of the median fluorescent intensities (MFIs) of CD11b and CD66b;https://www.nature.com/articles/s41598-022-07455-2

#For mature, circulating neutrophils, a phenotype of CD16hi, CXCR2hi, CXCR4low, and CD62Lhi has been reported.

#Neutrophil profiling illuminates anti-tumor antigen-presenting potency

#黄来源：CD101区分成熟与否,但不适合

In [154]:
adata.obs[groupby].value_counts()

L2_leiden_scVI_0.5
0    76111
3    48324
2    45233
1    44802
4    27359
5      567
6      108
Name: count, dtype: int64

In [155]:
check_dict_duplicates(cell_dict)

'无重复'

In [156]:
# Generate new assignments
for i in cell_dict.keys():
    ind = pd.Series(adata.obs[groupby]).isin(cell_dict[i])
    adata.obs.loc[ind,'Celltype_L1_L2_Refine'] = i

In [157]:
(adata.obs['Celltype_L1_L2_Refine'].isna()).value_counts()

Celltype_L1_L2_Refine
False    242504
Name: count, dtype: int64

In [158]:
adata.obs.loc[adata.obs['receptor_type']=='TCR','Celltype_L1_L2_Refine']="Doublet"
adata.obs.loc[adata.obs['receptor_type_BCR']=='BCR','Celltype_L1_L2_Refine']="Doublet"

In [159]:
adata.obs['Celltype_L1_L2_Refine'].value_counts()

Celltype_L1_L2_Refine
CEACAM8- Neutrophil    218825
Doublet                 23679
Name: count, dtype: int64

In [160]:
adata = adata[adata.obs['Celltype_L1_L2_Refine'] != "Doublet",:]

In [161]:
adata.write(f"{obj_path}{celltype}/{celltype}_L2_Refine.h5ad",compression="gzip")

# 13. pDC Refine

In [474]:
celltype="pDC"

In [475]:
adata = sc.read_h5ad(f"{obj_path}{celltype}/{celltype}_after_cluster_scVI.h5ad")
R_data_path = f"{obj_path}{celltype}"
sc.settings.figdir=f"{obj_path}{celltype}"

In [476]:
sc.pp.normalize_total(adata, target_sum=1e4)
sc.pp.log1p(adata)

In [477]:
del adata.obs['Celltype_L1_L2_Refine']

In [ ]:
sc.pl.umap(adata, color=['receptor_type','receptor_type_BCR'],
           frameon=False,
           legend_fontsize=4, legend_fontoutline=2, 
           size=4,
           legend_loc='on data')

In [479]:
pd.DataFrame(adata.obs[['L2_leiden_scVI_0.5', 'L2_leiden_scVI_1', 'L2_leiden_scVI_1.5', 'L2_leiden_scVI_2']]).to_csv(f'{R_data_path}/Level2_scVI_ForClustree.csv')
os.system(f'/home/liyanguo/anaconda3/envs/R/bin/Rscript ref_clustree.R {R_data_path} &')

0

Loading required package: ggraph
Loading required package: ggplot2


In [480]:
groupby = "L2_leiden_scVI_0.5"

In [481]:
xlsx = pd.ExcelWriter(f"{obj_path}{celltype}/{groupby}_freq_table.xlsx")
for group in ['SampleID', 'DonorID','scDblFinder.class','Phase',
              'Immune_All_High', 'Immune_All_Low', 'Adult_Human_Blood', 'Adult_Human_Bone_marrow',
              'AIFI_L1', 'AIFI_L2', 'AIFI_L3', 'predicted.celltype.l1', 'predicted.celltype.l2',
              'predicted.celltype.l3',
              'receptor_subtype', #TCR
              'VJ_1_v_call', 'VJ_1_j_call', 
              'VDJ_1_v_call', 'VDJ_1_j_call',
              'receptor_type_BCR', #BCR
              'VJ_1_v_call_BCR','VJ_1_j_call_BCR', 
              'VDJ_1_v_call_BCR', 'VDJ_1_j_call_BCR']:
    freq_table_multi = adata.obs.groupby([groupby, group]).size()
    pd.DataFrame(freq_table_multi).to_excel(xlsx,sheet_name=group)
xlsx.close()

In [ ]:
#数据簇间信息比较
fig, axs = plt.subplots(5, 2, figsize=(18, 12),constrained_layout=True)
plt.subplots_adjust(hspace=1,wspace=1)
sc.pl.violin(adata, keys='nCount_RNA', groupby=groupby, rotation=90,size=0.1,palette="pastel",linewidth=0,ax=axs[0,0], show=False)
sc.pl.violin(adata, keys='nFeature_RNA', groupby=groupby, rotation=90,size=0.1,palette="pastel",linewidth=0,ax=axs[0,1], show=False)
sc.pl.violin(adata, keys='log10GenesPerUMI', groupby=groupby, rotation=90,size=0.1,palette="pastel",linewidth=0,ax=axs[1,0], show=False)
sc.pl.violin(adata, keys='pct_counts_in_top_20_genes', groupby=groupby, rotation=90,size=0.1,palette="pastel",linewidth=0,ax=axs[1,1], show=False)

sc.pl.violin(adata, keys='percent_apop', groupby=groupby, rotation=90,size=0.1,palette="pastel",linewidth=0,ax=axs[2,0], show=False)
sc.pl.violin(adata, keys='percent_ribo', groupby=groupby, rotation=90,size=0.1,palette="pastel",linewidth=0,ax=axs[2,1], show=False)
sc.pl.violin(adata, keys='percent_ieg', groupby=groupby, rotation=90,size=0.1,palette="pastel",linewidth=0,ax=axs[3,0], show=False)
sc.pl.violin(adata, keys='percent_oxphos', groupby=groupby, rotation=90,size=0.1,palette="pastel",linewidth=0,ax=axs[3,1], show=False)
sc.pl.violin(adata, keys='percent_hemo', groupby=groupby, rotation=90,size=0.1,palette="pastel",linewidth=0,ax=axs[4,0], show=False)
sc.pl.violin(adata, keys='G2M.Score', groupby=groupby, rotation=90,size=0.1,palette="pastel",linewidth=0,ax=axs[4,1], show=False)

fig.tight_layout()
plt.savefig(f'{obj_path}{celltype}/{groupby}_scVI_L2_metadata.png')

adata.obs['cluster_dummy'] = adata.obs[groupby] == adata.obs[groupby].cat.categories[17]
sc.pl.umap(adata, color='cluster_dummy',legend_fontsize=4, legend_fontoutline=2,size=4,legend_loc='on data')

In [483]:
sc.tl.dendrogram(adata,groupby=groupby,use_rep='X_scVI')#

In [ ]:
#cosg差异
cosg.cosg(adata, key_added=f'cosg_{groupby}', groupby=groupby,
          mu=10,n_genes_user=100,remove_lowly_expressed=True,
         )
df_tmp = pd.DataFrame(adata.uns[f'cosg_{groupby}']['names'])
df_tmp.to_csv(f"{obj_path}{celltype}/cosg_{groupby}.csv")
#cosg差异作图
df_tmp=pd.DataFrame(adata.uns[f'cosg_{groupby}']['names'][:8,]).T
df_tmp=df_tmp.reindex(adata.uns['dendrogram_'+groupby]['categories_ordered'])
marker_genes_list={idx: list(row.values) for idx, row in df_tmp.iterrows()}
marker_genes_list = {k: v for k, v in marker_genes_list.items() if not any(isinstance(x, float) for x in v)}
sc.pl.dotplot(adata, marker_genes_list,
             groupby=groupby,
             dendrogram=True,
             swap_axes=False,
             standard_scale='var',
             save=f'cosg_{groupby}',
             cmap='Spectral_r')

In [485]:
FeatureMatrix = pd.DataFrame(adata.uns[f'cosg_{groupby}']['names'])

In [ ]:
%%R -i FeatureMatrix,R_data_path,groupby
source('GetEnrich.R')
GetEnrich(FeatureMatrix,plot_term_number = 10,R_data_path=R_data_path,groupby=groupby,
          gmtfile="/localdisk/immune/Marker/msigdb_v2024.1.Hs_GMTs/h.all.v2024.1.Hs.symbols.gmt")
#"/localdisk/immune/Marker/msigdb_v2024.1.Hs_GMTs/h.all.v2024.1.Hs.symbols.gmt"
#"/localdisk/immune/Marker/Immunity_PBMC.gmt"
#"/localdisk/immune/Marker/sepsis_PBMC.gmt"
#"/localdisk/immune/Marker/msigdb_v2024.1.Hs_GMTs/c5.go.bp.v2024.1.Hs.symbols.gmt"
#"/localdisk/immune/Marker/msigdb_v2024.1.Hs_GMTs/c2.cp.kegg_legacy.v2024.1.Hs.symbols.gmt"

In [ ]:
fig, axs = plt.subplots(ncols=4, nrows=2, figsize=(16, 8))
sc.pl.umap(adata, color=groupby,legend_fontsize=4, legend_fontoutline=2,size=4,legend_loc='on data',ax=axs[0,0],show=False)
sc.pl.umap(adata, color='CLEC4C',legend_fontsize=4, legend_fontoutline=2,size=4,legend_loc='on data',ax=axs[0,1],show=False)
sc.pl.umap(adata, color='IL3RA',legend_fontsize=4, legend_fontoutline=2,size=4,legend_loc='on data',ax=axs[0,2],show=False)
sc.pl.umap(adata, color='CD7',legend_fontsize=4, legend_fontoutline=2,size=4,legend_loc='on data',ax=axs[0,3],show=False)

sc.pl.violin(adata,'ETV6',groupby=groupby,show=False, ax=axs[1,0],size=0)
sc.pl.violin(adata,'RUNX1',groupby=groupby,show=False, ax=axs[1,1],size=0)
sc.pl.violin(adata,'FLT3',groupby=groupby,show=False, ax=axs[1,2],size=0)
sc.pl.violin(adata,'IL6ST',groupby=groupby,show=False, ax=axs[1,3],size=0)
plt.show()

In [ ]:
sc.pl.violin(adata,'IL3RA',groupby=groupby,size=0)

In [489]:
cell_dict = {'Doublet':['6',#Neu
                        '9',
                       '2','8'],#TNK
             
             "pDCs":['0','1','3','4','5','7',],
            }

In [490]:
adata.obs[groupby].value_counts()

L2_leiden_scVI_0.5
3    495
4    447
5    372
2    332
7    241
1    232
0    146
6    144
8     67
9     16
Name: count, dtype: int64

In [491]:
check_dict_duplicates(cell_dict)

'无重复'

In [492]:
# Generate new assignments
for i in cell_dict.keys():
    ind = pd.Series(adata.obs[groupby]).isin(cell_dict[i])
    adata.obs.loc[ind,'Celltype_L1_L2_Refine'] = i

In [493]:
(adata.obs['Celltype_L1_L2_Refine'].isna()).value_counts()

Celltype_L1_L2_Refine
False    2492
Name: count, dtype: int64

In [494]:
adata.obs['Celltype_L1_L2_Refine'].value_counts()

Celltype_L1_L2_Refine
pDC        1933
Doublet     559
Name: count, dtype: int64

In [495]:
adata.obs.loc[adata.obs['receptor_type']=='TCR','Celltype_L1_L2_Refine']="Doublet"
adata.obs.loc[adata.obs['receptor_type_BCR']=='BCR','Celltype_L1_L2_Refine']="Doublet"

In [496]:
adata.obs['Celltype_L1_L2_Refine'].value_counts()

Celltype_L1_L2_Refine
pDC        1699
Doublet     793
Name: count, dtype: int64

In [497]:
adata = adata[adata.obs['Celltype_L1_L2_Refine'] != "Doublet",:]

In [498]:
adata.write(f"{obj_path}{celltype}/{celltype}_L2_Refine.h5ad",compression="gzip")

# 14. Dendritic Refine

In [499]:
celltype="Dendritic"

In [500]:
adata = sc.read_h5ad(f"{obj_path}{celltype}/{celltype}_after_cluster_scVI.h5ad")
R_data_path = f"{obj_path}{celltype}"
sc.settings.figdir=f"{obj_path}{celltype}"

In [501]:
sc.pp.normalize_total(adata, target_sum=1e4)
sc.pp.log1p(adata)

In [502]:
del adata.obs['Celltype_L1_L2_Refine']

In [ ]:
sc.pl.umap(adata, color=['receptor_type','receptor_type_BCR'],
           frameon=False,
           legend_fontsize=4, legend_fontoutline=2, 
           size=4,
           legend_loc='on data')

In [504]:
pd.DataFrame(adata.obs[['L2_leiden_scVI_0.5', 'L2_leiden_scVI_1', 'L2_leiden_scVI_1.5', 'L2_leiden_scVI_2']]).to_csv(f'{R_data_path}/Level2_scVI_ForClustree.csv')
os.system(f'/home/liyanguo/anaconda3/envs/R/bin/Rscript ref_clustree.R {R_data_path} &')

0

Loading required package: ggraph
Loading required package: ggplot2


In [505]:
groupby = "L2_leiden_scVI_0.5"

In [506]:
xlsx = pd.ExcelWriter(f"{obj_path}{celltype}/{groupby}_freq_table.xlsx")
for group in ['SampleID', 'DonorID','scDblFinder.class','Phase',
              'Immune_All_High', 'Immune_All_Low', 'Adult_Human_Blood', 'Adult_Human_Bone_marrow',
              'AIFI_L1', 'AIFI_L2', 'AIFI_L3', 'predicted.celltype.l1', 'predicted.celltype.l2',
              'predicted.celltype.l3',
              'receptor_subtype', #TCR
              'VJ_1_v_call', 'VJ_1_j_call', 
              'VDJ_1_v_call', 'VDJ_1_j_call',
              'receptor_type_BCR', #BCR
              'VJ_1_v_call_BCR','VJ_1_j_call_BCR', 
              'VDJ_1_v_call_BCR', 'VDJ_1_j_call_BCR']:
    freq_table_multi = adata.obs.groupby([groupby, group]).size()
    pd.DataFrame(freq_table_multi).to_excel(xlsx,sheet_name=group)
xlsx.close()

In [ ]:
#数据簇间信息比较
fig, axs = plt.subplots(5, 2, figsize=(18, 12),constrained_layout=True)
plt.subplots_adjust(hspace=1,wspace=1)
sc.pl.violin(adata, keys='nCount_RNA', groupby=groupby, rotation=90,size=0.1,palette="pastel",linewidth=0,ax=axs[0,0], show=False)
sc.pl.violin(adata, keys='nFeature_RNA', groupby=groupby, rotation=90,size=0.1,palette="pastel",linewidth=0,ax=axs[0,1], show=False)
sc.pl.violin(adata, keys='log10GenesPerUMI', groupby=groupby, rotation=90,size=0.1,palette="pastel",linewidth=0,ax=axs[1,0], show=False)
sc.pl.violin(adata, keys='pct_counts_in_top_20_genes', groupby=groupby, rotation=90,size=0.1,palette="pastel",linewidth=0,ax=axs[1,1], show=False)

sc.pl.violin(adata, keys='percent_apop', groupby=groupby, rotation=90,size=0.1,palette="pastel",linewidth=0,ax=axs[2,0], show=False)
sc.pl.violin(adata, keys='percent_ribo', groupby=groupby, rotation=90,size=0.1,palette="pastel",linewidth=0,ax=axs[2,1], show=False)
sc.pl.violin(adata, keys='percent_ieg', groupby=groupby, rotation=90,size=0.1,palette="pastel",linewidth=0,ax=axs[3,0], show=False)
sc.pl.violin(adata, keys='percent_oxphos', groupby=groupby, rotation=90,size=0.1,palette="pastel",linewidth=0,ax=axs[3,1], show=False)
sc.pl.violin(adata, keys='percent_hemo', groupby=groupby, rotation=90,size=0.1,palette="pastel",linewidth=0,ax=axs[4,0], show=False)
sc.pl.violin(adata, keys='G2M.Score', groupby=groupby, rotation=90,size=0.1,palette="pastel",linewidth=0,ax=axs[4,1], show=False)

fig.tight_layout()
plt.savefig(f'{obj_path}{celltype}/{groupby}_scVI_L2_metadata.png')

adata.obs['cluster_dummy'] = adata.obs[groupby] == adata.obs[groupby].cat.categories[17]
sc.pl.umap(adata, color='cluster_dummy',legend_fontsize=4, legend_fontoutline=2,size=4,legend_loc='on data')

In [508]:
sc.tl.dendrogram(adata,groupby=groupby,use_rep='X_scVI')#

In [ ]:
#cosg差异
cosg.cosg(adata, key_added=f'cosg_{groupby}', groupby=groupby,
          mu=10,n_genes_user=100,remove_lowly_expressed=True,
         )
df_tmp = pd.DataFrame(adata.uns[f'cosg_{groupby}']['names'])
df_tmp.to_csv(f"{obj_path}{celltype}/cosg_{groupby}.csv")
#cosg差异作图
df_tmp=pd.DataFrame(adata.uns[f'cosg_{groupby}']['names'][:8,]).T
df_tmp=df_tmp.reindex(adata.uns['dendrogram_'+groupby]['categories_ordered'])
marker_genes_list={idx: list(row.values) for idx, row in df_tmp.iterrows()}
marker_genes_list = {k: v for k, v in marker_genes_list.items() if not any(isinstance(x, float) for x in v)}
sc.pl.dotplot(adata, marker_genes_list,
             groupby=groupby,
             dendrogram=True,
             swap_axes=False,
             standard_scale='var',
             save=f'cosg_{groupby}',
             cmap='Spectral_r')

In [ ]:
fig, axs = plt.subplots(ncols=4, nrows=2, figsize=(16, 8))
sc.pl.umap(adata, color=groupby,legend_fontsize=4, legend_fontoutline=2,size=4,legend_loc='on data',ax=axs[0,0],show=False)
sc.pl.umap(adata, color='CD1C',legend_fontsize=4, legend_fontoutline=2,size=4,legend_loc='on data',ax=axs[0,1],show=False)
sc.pl.umap(adata, color='CD14',legend_fontsize=4, legend_fontoutline=2,size=4,legend_loc='on data',ax=axs[0,2],show=False)
sc.pl.umap(adata, color='FCGR3B',legend_fontsize=4, legend_fontoutline=2,size=4,legend_loc='on data',ax=axs[0,3],show=False)

sc.pl.violin(adata,'CSF3R',groupby=groupby,show=False, ax=axs[1,0],size=0)
sc.pl.violin(adata,'FCGR3B',groupby=groupby,show=False, ax=axs[1,1],size=0)
sc.pl.violin(adata,'CLEC9A',groupby=groupby,show=False, ax=axs[1,2],size=0)
sc.pl.violin(adata,'CLEC10A',groupby=groupby,show=False, ax=axs[1,3],size=0)
plt.show()

In [ ]:
sc.pl.violin(adata,'CD209',groupby=groupby,size=0)

In [512]:
FeatureMatrix = pd.DataFrame(adata.uns[f'cosg_{groupby}']['names'])

In [ ]:
%%R -i FeatureMatrix,R_data_path,groupby
source('GetEnrich.R')
GetEnrich(FeatureMatrix,plot_term_number = 10,R_data_path=R_data_path,groupby=groupby,
          gmtfile="/localdisk/immune/Marker/msigdb_v2024.1.Hs_GMTs/h.all.v2024.1.Hs.symbols.gmt")
#"/localdisk/immune/Marker/msigdb_v2024.1.Hs_GMTs/h.all.v2024.1.Hs.symbols.gmt"
#"/localdisk/immune/Marker/Immunity_PBMC.gmt"
#"/localdisk/immune/Marker/sepsis_PBMC.gmt"
#"/localdisk/immune/Marker/msigdb_v2024.1.Hs_GMTs/c5.go.bp.v2024.1.Hs.symbols.gmt"
#"/localdisk/immune/Marker/msigdb_v2024.1.Hs_GMTs/c2.cp.kegg_legacy.v2024.1.Hs.symbols.gmt"

In [523]:
cell_dict = {'Doublet':['6',# Neu + mono + scDblFinder
                        '4',#NK
                        '10',#Neu
                        '8',#T
                       ],
             'cDCs':['0','1','2','3','5','7','9',
                          '11'],
            }
#cDC2 ,CD1C CLEC10A、IRF2、IRF4、RelB、RBP-J	人：CD11c, HLA-DR, CD1c (BDCA1), CD11b，FCER1A, CD2, CD172A, ILT1
#cDC1,XCR1 BATF3 IRF8, 人：CD11c, HLA-DR, CD141 (BCDA3) CLEC9A、CADM1、THBD 
# moDC	KLF4, 人：CD11c, CD11b, CD1a, CD1c (BDCA‐1),其他：CD206, CD209, CD172A

In [524]:
adata.obs[groupby].value_counts()

L2_leiden_scVI_0.5
1     717
2     609
0     602
6     500
3     448
9     424
10    336
7     308
5     245
8     244
4     179
11    118
Name: count, dtype: int64

In [525]:
check_dict_duplicates(cell_dict)

'无重复'

In [526]:
# Generate new assignments
for i in cell_dict.keys():
    ind = pd.Series(adata.obs[groupby]).isin(cell_dict[i])
    adata.obs.loc[ind,'Celltype_L1_L2_Refine'] = i

In [527]:
(adata.obs['Celltype_L1_L2_Refine'].isna()).value_counts()

Celltype_L1_L2_Refine
False    4730
Name: count, dtype: int64

In [528]:
adata.obs.loc[adata.obs['receptor_type']=='TCR','Celltype_L1_L2_Refine']="Doublet"
adata.obs.loc[adata.obs['receptor_type_BCR']=='BCR','Celltype_L1_L2_Refine']="Doublet"

In [529]:
adata.obs['Celltype_L1_L2_Refine'].value_counts()

Celltype_L1_L2_Refine
Dendritic    3096
Doublet      1634
Name: count, dtype: int64

In [530]:
adata = adata[adata.obs['Celltype_L1_L2_Refine'] != "Doublet",:]

In [531]:
adata.write(f"{obj_path}{celltype}/{celltype}_L2_Refine.h5ad",compression="gzip")

# 15. Platelet Refine

In [532]:
celltype="Platelet"

In [533]:
adata = sc.read_h5ad(f"{obj_path}{celltype}/{celltype}_after_cluster_scVI.h5ad")
R_data_path = f"{obj_path}{celltype}"
sc.settings.figdir=f"{obj_path}{celltype}"

In [534]:
sc.pp.normalize_total(adata, target_sum=1e4)
sc.pp.log1p(adata)

In [535]:
del adata.obs['Celltype_L1_L2_Refine']

In [ ]:
sc.pl.umap(adata, color=['receptor_type','receptor_type_BCR'],
           frameon=False,
           legend_fontsize=4, legend_fontoutline=2, 
           size=4,
           legend_loc='on data')

In [537]:
pd.DataFrame(adata.obs[['L2_leiden_scVI_0.5', 'L2_leiden_scVI_1', 'L2_leiden_scVI_1.5', 'L2_leiden_scVI_2']]).to_csv(f'{R_data_path}/Level2_scVI_ForClustree.csv')
os.system(f'/home/liyanguo/anaconda3/envs/R/bin/Rscript ref_clustree.R {R_data_path} &')

0

In [538]:
groupby = "L2_leiden_scVI_0.5"

In [539]:
xlsx = pd.ExcelWriter(f"{obj_path}{celltype}/{groupby}_freq_table.xlsx")
for group in ['SampleID', 'DonorID','scDblFinder.class','Phase',
              'Immune_All_High', 'Immune_All_Low', 'Adult_Human_Blood', 'Adult_Human_Bone_marrow',
              'AIFI_L1', 'AIFI_L2', 'AIFI_L3', 'predicted.celltype.l1', 'predicted.celltype.l2',
              'predicted.celltype.l3',
              'receptor_subtype', #TCR
              'VJ_1_v_call', 'VJ_1_j_call', 
              'VDJ_1_v_call', 'VDJ_1_j_call',
              'receptor_type_BCR', #BCR
              'VJ_1_v_call_BCR','VJ_1_j_call_BCR', 
              'VDJ_1_v_call_BCR', 'VDJ_1_j_call_BCR']:
    freq_table_multi = adata.obs.groupby([groupby, group]).size()
    pd.DataFrame(freq_table_multi).to_excel(xlsx,sheet_name=group)
xlsx.close()

Loading required package: ggraph
Loading required package: ggplot2


In [ ]:
#数据簇间信息比较
fig, axs = plt.subplots(5, 2, figsize=(18, 12),constrained_layout=True)
plt.subplots_adjust(hspace=1,wspace=1)
sc.pl.violin(adata, keys='nCount_RNA', groupby=groupby, rotation=90,size=0.1,palette="pastel",linewidth=0,ax=axs[0,0], show=False)
sc.pl.violin(adata, keys='nFeature_RNA', groupby=groupby, rotation=90,size=0.1,palette="pastel",linewidth=0,ax=axs[0,1], show=False)
sc.pl.violin(adata, keys='log10GenesPerUMI', groupby=groupby, rotation=90,size=0.1,palette="pastel",linewidth=0,ax=axs[1,0], show=False)
sc.pl.violin(adata, keys='pct_counts_in_top_20_genes', groupby=groupby, rotation=90,size=0.1,palette="pastel",linewidth=0,ax=axs[1,1], show=False)

sc.pl.violin(adata, keys='percent_apop', groupby=groupby, rotation=90,size=0.1,palette="pastel",linewidth=0,ax=axs[2,0], show=False)
sc.pl.violin(adata, keys='percent_ribo', groupby=groupby, rotation=90,size=0.1,palette="pastel",linewidth=0,ax=axs[2,1], show=False)
sc.pl.violin(adata, keys='percent_ieg', groupby=groupby, rotation=90,size=0.1,palette="pastel",linewidth=0,ax=axs[3,0], show=False)
sc.pl.violin(adata, keys='percent_oxphos', groupby=groupby, rotation=90,size=0.1,palette="pastel",linewidth=0,ax=axs[3,1], show=False)
sc.pl.violin(adata, keys='percent_hemo', groupby=groupby, rotation=90,size=0.1,palette="pastel",linewidth=0,ax=axs[4,0], show=False)
sc.pl.violin(adata, keys='G2M.Score', groupby=groupby, rotation=90,size=0.1,palette="pastel",linewidth=0,ax=axs[4,1], show=False)

fig.tight_layout()
plt.savefig(f'{obj_path}{celltype}/{groupby}_scVI_L2_metadata.png')

adata.obs['cluster_dummy'] = adata.obs[groupby] == adata.obs[groupby].cat.categories[17]
sc.pl.umap(adata, color='cluster_dummy',legend_fontsize=4, legend_fontoutline=2,size=4,legend_loc='on data')

In [541]:
sc.tl.dendrogram(adata,groupby=groupby,use_rep='X_scVI')#

In [ ]:
#cosg差异
cosg.cosg(adata, key_added=f'cosg_{groupby}', groupby=groupby,
          mu=10,n_genes_user=100,remove_lowly_expressed=True,
         )
df_tmp = pd.DataFrame(adata.uns[f'cosg_{groupby}']['names'])
df_tmp.to_csv(f"{obj_path}{celltype}/cosg_{groupby}.csv")
#cosg差异作图
df_tmp=pd.DataFrame(adata.uns[f'cosg_{groupby}']['names'][:8,]).T
df_tmp=df_tmp.reindex(adata.uns['dendrogram_'+groupby]['categories_ordered'])
marker_genes_list={idx: list(row.values) for idx, row in df_tmp.iterrows()}
marker_genes_list = {k: v for k, v in marker_genes_list.items() if not any(isinstance(x, float) for x in v)}
sc.pl.dotplot(adata, marker_genes_list,
             groupby=groupby,
             dendrogram=True,
             swap_axes=False,
             standard_scale='var',
             save=f'cosg_{groupby}',
             cmap='Spectral_r')

In [543]:
FeatureMatrix = pd.DataFrame(adata.uns[f'cosg_{groupby}']['names'])

In [ ]:
%%R -i FeatureMatrix,R_data_path,groupby
source('GetEnrich.R')
GetEnrich(FeatureMatrix,plot_term_number = 10,R_data_path=R_data_path,groupby=groupby,
          gmtfile="/localdisk/immune/Marker/msigdb_v2024.1.Hs_GMTs/h.all.v2024.1.Hs.symbols.gmt")
#"/localdisk/immune/Marker/msigdb_v2024.1.Hs_GMTs/h.all.v2024.1.Hs.symbols.gmt"
#"/localdisk/immune/Marker/Immunity_PBMC.gmt"
#"/localdisk/immune/Marker/sepsis_PBMC.gmt"
#"/localdisk/immune/Marker/msigdb_v2024.1.Hs_GMTs/c5.go.bp.v2024.1.Hs.symbols.gmt"
#"/localdisk/immune/Marker/msigdb_v2024.1.Hs_GMTs/c2.cp.kegg_legacy.v2024.1.Hs.symbols.gmt"

In [ ]:
fig, axs = plt.subplots(ncols=4, nrows=2, figsize=(16, 8))
sc.pl.umap(adata, color=groupby,legend_fontsize=4, legend_fontoutline=2,size=4,legend_loc='on data',ax=axs[0,0],show=False)
sc.pl.umap(adata, color='CD14',legend_fontsize=4, legend_fontoutline=2,size=4,legend_loc='on data',ax=axs[0,1],show=False)
sc.pl.umap(adata, color='FCGR3B',legend_fontsize=4, legend_fontoutline=2,size=4,legend_loc='on data',ax=axs[0,2],show=False)
sc.pl.umap(adata, color='CD2',legend_fontsize=4, legend_fontoutline=2,size=4,legend_loc='on data',ax=axs[0,3],show=False)

sc.pl.violin(adata,'FCER1G',groupby=groupby,show=False, ax=axs[1,0],size=0)
sc.pl.violin(adata,'PRF1',groupby=groupby,show=False, ax=axs[1,1],size=0)
sc.pl.violin(adata,'HLA-DRA',groupby=groupby,show=False, ax=axs[1,2],size=0)
sc.pl.violin(adata,'PTPRC',groupby=groupby,show=False, ax=axs[1,3],size=0)
plt.show()

In [ ]:
sc.pl.violin(adata,'',groupby=groupby,size=0)

In [547]:
cell_dict = {'Doublet':['0',#Mono
                        '11',#Mono
                        '10',#B
                        '4','9',#NK
                        '5','6','7','8',#T
                        '1','3',#Neu
                       ],
             'Platelets':['2',],
            }

In [548]:
check_dict_duplicates(cell_dict)

'无重复'

In [549]:
# Generate new assignments
for i in cell_dict.keys():
    ind = pd.Series(adata.obs[groupby]).isin(cell_dict[i])
    adata.obs.loc[ind,'Celltype_L1_L2_Refine'] = i

In [550]:
(adata.obs['Celltype_L1_L2_Refine'].isna()).value_counts()

Celltype_L1_L2_Refine
False    12348
Name: count, dtype: int64

In [551]:
adata.obs[groupby].value_counts()

L2_leiden_scVI_0.5
4     2406
7     1960
1     1724
2     1379
5     1351
0     1241
3      832
8      564
6      512
10     165
11     126
9       88
Name: count, dtype: int64

In [552]:
adata.obs.loc[adata.obs['receptor_type']=='TCR','Celltype_L1_L2_Refine']="Doublet"
adata.obs.loc[adata.obs['receptor_type_BCR']=='BCR','Celltype_L1_L2_Refine']="Doublet"

In [553]:
adata.obs['Celltype_L1_L2_Refine'].value_counts()

Celltype_L1_L2_Refine
Doublet     11159
Platelet     1189
Name: count, dtype: int64

In [554]:
adata = adata[adata.obs['Celltype_L1_L2_Refine'] != "Doublet",:]

In [555]:
adata.write(f"{obj_path}{celltype}/{celltype}_L2_Refine.h5ad",compression="gzip")

# 16. Proliferative_TNK Refine

In [556]:
celltype="Proliferative_TNK"

In [557]:
adata = sc.read_h5ad(f"{obj_path}{celltype}/{celltype}_after_cluster_scVI.h5ad")
R_data_path = f"{obj_path}{celltype}"
sc.settings.figdir=f"{obj_path}{celltype}"

In [558]:
sc.pp.normalize_total(adata, target_sum=1e4)
sc.pp.log1p(adata)

In [559]:
del adata.obs['Celltype_L1_L2_Refine']

In [ ]:
sc.pl.umap(adata, color=['receptor_type','receptor_type_BCR'],
           frameon=False,
           legend_fontsize=4, legend_fontoutline=2, 
           size=4,
           legend_loc='on data')

In [561]:
pd.DataFrame(adata.obs[['L2_leiden_scVI_0.5', 'L2_leiden_scVI_1', 'L2_leiden_scVI_1.5', 'L2_leiden_scVI_2']]).to_csv(f'{R_data_path}/Level2_scVI_ForClustree.csv')
os.system(f'/home/liyanguo/anaconda3/envs/R/bin/Rscript ref_clustree.R {R_data_path} &')

0

In [562]:
groupby = "L2_leiden_scVI_0.5"

Loading required package: ggraph
Loading required package: ggplot2


In [563]:
xlsx = pd.ExcelWriter(f"{obj_path}{celltype}/{groupby}_freq_table.xlsx")
for group in ['SampleID', 'DonorID','scDblFinder.class','Phase',
              'Immune_All_High', 'Immune_All_Low', 'Adult_Human_Blood', 'Adult_Human_Bone_marrow',
              'AIFI_L1', 'AIFI_L2', 'AIFI_L3', 'predicted.celltype.l1', 'predicted.celltype.l2',
              'predicted.celltype.l3',
              'receptor_subtype', #TCR
              'VJ_1_v_call', 'VJ_1_j_call', 
              'VDJ_1_v_call', 'VDJ_1_j_call',
              'receptor_type_BCR', #BCR
              'VJ_1_v_call_BCR','VJ_1_j_call_BCR', 
              'VDJ_1_v_call_BCR', 'VDJ_1_j_call_BCR']:
    freq_table_multi = adata.obs.groupby([groupby, group]).size()
    pd.DataFrame(freq_table_multi).to_excel(xlsx,sheet_name=group)
xlsx.close()

In [ ]:
#数据簇间信息比较
fig, axs = plt.subplots(5, 2, figsize=(18, 12),constrained_layout=True)
plt.subplots_adjust(hspace=1,wspace=1)
sc.pl.violin(adata, keys='nCount_RNA', groupby=groupby, rotation=90,size=0.1,palette="pastel",linewidth=0,ax=axs[0,0], show=False)
sc.pl.violin(adata, keys='nFeature_RNA', groupby=groupby, rotation=90,size=0.1,palette="pastel",linewidth=0,ax=axs[0,1], show=False)
sc.pl.violin(adata, keys='log10GenesPerUMI', groupby=groupby, rotation=90,size=0.1,palette="pastel",linewidth=0,ax=axs[1,0], show=False)
sc.pl.violin(adata, keys='pct_counts_in_top_20_genes', groupby=groupby, rotation=90,size=0.1,palette="pastel",linewidth=0,ax=axs[1,1], show=False)

sc.pl.violin(adata, keys='percent_apop', groupby=groupby, rotation=90,size=0.1,palette="pastel",linewidth=0,ax=axs[2,0], show=False)
sc.pl.violin(adata, keys='percent_ribo', groupby=groupby, rotation=90,size=0.1,palette="pastel",linewidth=0,ax=axs[2,1], show=False)
sc.pl.violin(adata, keys='percent_ieg', groupby=groupby, rotation=90,size=0.1,palette="pastel",linewidth=0,ax=axs[3,0], show=False)
sc.pl.violin(adata, keys='percent_oxphos', groupby=groupby, rotation=90,size=0.1,palette="pastel",linewidth=0,ax=axs[3,1], show=False)
sc.pl.violin(adata, keys='percent_hemo', groupby=groupby, rotation=90,size=0.1,palette="pastel",linewidth=0,ax=axs[4,0], show=False)
sc.pl.violin(adata, keys='G2M.Score', groupby=groupby, rotation=90,size=0.1,palette="pastel",linewidth=0,ax=axs[4,1], show=False)

fig.tight_layout()
plt.savefig(f'{obj_path}{celltype}/{groupby}_scVI_L2_metadata.png')

adata.obs['cluster_dummy'] = adata.obs[groupby] == adata.obs[groupby].cat.categories[17]
sc.pl.umap(adata, color='cluster_dummy',legend_fontsize=4, legend_fontoutline=2,size=4,legend_loc='on data')

In [565]:
sc.tl.dendrogram(adata,groupby=groupby,use_rep='X_scVI')#

In [ ]:
#cosg差异
cosg.cosg(adata, key_added=f'cosg_{groupby}', groupby=groupby,
          mu=10,n_genes_user=100,remove_lowly_expressed=True,
         )
df_tmp = pd.DataFrame(adata.uns[f'cosg_{groupby}']['names'])
df_tmp.to_csv(f"{obj_path}{celltype}/cosg_{groupby}.csv")
#cosg差异作图
df_tmp=pd.DataFrame(adata.uns[f'cosg_{groupby}']['names'][:8,]).T
df_tmp=df_tmp.reindex(adata.uns['dendrogram_'+groupby]['categories_ordered'])
marker_genes_list={idx: list(row.values) for idx, row in df_tmp.iterrows()}
marker_genes_list = {k: v for k, v in marker_genes_list.items() if not any(isinstance(x, float) for x in v)}
sc.pl.dotplot(adata, marker_genes_list,
             groupby=groupby,
             dendrogram=True,
             swap_axes=False,
             standard_scale='var',
             save=f'cosg_{groupby}',
             cmap='Spectral_r')

In [567]:
FeatureMatrix = pd.DataFrame(adata.uns[f'cosg_{groupby}']['names'])

In [ ]:
%%R -i FeatureMatrix,R_data_path,groupby
source('GetEnrich.R')
GetEnrich(FeatureMatrix,plot_term_number = 10,R_data_path=R_data_path,groupby=groupby,
          gmtfile="/localdisk/immune/Marker/msigdb_v2024.1.Hs_GMTs/h.all.v2024.1.Hs.symbols.gmt")
#"/localdisk/immune/Marker/msigdb_v2024.1.Hs_GMTs/h.all.v2024.1.Hs.symbols.gmt"
#"/localdisk/immune/Marker/Immunity_PBMC.gmt"
#"/localdisk/immune/Marker/sepsis_PBMC.gmt"
#"/localdisk/immune/Marker/msigdb_v2024.1.Hs_GMTs/c5.go.bp.v2024.1.Hs.symbols.gmt"
#"/localdisk/immune/Marker/msigdb_v2024.1.Hs_GMTs/c2.cp.kegg_legacy.v2024.1.Hs.symbols.gmt"

In [ ]:
fig, axs = plt.subplots(ncols=4, nrows=2, figsize=(16, 8))
sc.pl.umap(adata, color=groupby,legend_fontsize=4, legend_fontoutline=2,size=4,legend_loc='on data',ax=axs[0,0],show=False)
sc.pl.umap(adata, color='CD14',legend_fontsize=4, legend_fontoutline=2,size=4,legend_loc='on data',ax=axs[0,1],show=False)
sc.pl.umap(adata, color='FCGR3B',legend_fontsize=4, legend_fontoutline=2,size=4,legend_loc='on data',ax=axs[0,2],show=False)
sc.pl.umap(adata, color='MKI67',legend_fontsize=4, legend_fontoutline=2,size=4,legend_loc='on data',ax=axs[0,3],show=False)

sc.pl.violin(adata,'SPON2',groupby=groupby,show=False, ax=axs[1,0],size=0)
sc.pl.violin(adata,'CD3D',groupby=groupby,show=False, ax=axs[1,1],size=0)
sc.pl.violin(adata,'CD4',groupby=groupby,show=False, ax=axs[1,2],size=0)
sc.pl.violin(adata,'CD8A',groupby=groupby,show=False, ax=axs[1,3],size=0)
plt.show()

In [ ]:
sc.pl.violin(adata,'',groupby=groupby,size=0)

In [570]:
cell_dict = {'Doublet':['9',#Neu
                       ],
             'Proliferative T/NK':['0','1','2','3','4','5','6','7','8'],
            }

In [571]:
adata.obs[groupby].value_counts()

L2_leiden_scVI_0.5
2    410
3    345
1    311
6    285
8    279
7    240
0    205
4    193
5    131
9    130
Name: count, dtype: int64

In [572]:
check_dict_duplicates(cell_dict)

'无重复'

In [573]:
# Generate new assignments
for i in cell_dict.keys():
    ind = pd.Series(adata.obs[groupby]).isin(cell_dict[i])
    adata.obs.loc[ind,'Celltype_L1_L2_Refine'] = i

In [574]:
(adata.obs['Celltype_L1_L2_Refine'].isna()).value_counts()

Celltype_L1_L2_Refine
False    2529
Name: count, dtype: int64

In [575]:
adata.obs.loc[adata.obs['receptor_type_BCR']=='BCR','Celltype_L1_L2_Refine']="Doublet"

In [576]:
adata.obs['Celltype_L1_L2_Refine'].value_counts()

Celltype_L1_L2_Refine
Proliferative T/NK    2341
Doublet                188
Name: count, dtype: int64

In [577]:
adata = adata[adata.obs['Celltype_L1_L2_Refine'] != "Doublet",:]

In [578]:
adata.write(f"{obj_path}{celltype}/{celltype}_L2_Refine.h5ad",compression="gzip")

# 16. 合并所有数据

In [24]:
celltypes=['CEACAM8_Neg_Neutrophil','CEACAM8_Pos_Neutrophil','Monocyte',
           'CD4T','NK','CD8T',
           'gdT','B','MAIT',
           'Basophil','Dendritic','pDC',
           'Platelet',
           'HSPC','Proliferative_TNK','iNKT']

In [25]:
adatas= {}
for i in tqdm(celltypes):
    adata = sc.read_h5ad(f"{obj_path}{i}/{i}_L2_Refine.h5ad")
    adatas[i] = adata

100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 16/16 [00:36<00:00,  2.30s/it]


In [26]:
adata = ad.concat(adatas)

In [27]:
adata

AnnData object with n_obs × n_vars = 717723 × 38606
    obs: 'orig.ident', 'nCount_RNA', 'nFeature_RNA', 'SampleID', 'DonorID', 'percent_mito', 'percent_ribo', 'percent_mito_ribo', 'log10GenesPerUMI', 'percent_top50', 'percent_oxphos', 'percent_apop', 'percent_dna_repair', 'percent_ieg', 'percent_hemo', 'S.Score', 'G2M.Score', 'Phase', 'Immune_All_High', 'Immune_All_Low', 'Adult_Human_Blood', 'Adult_Human_Bone_marrow', 'AIFI_L1', 'AIFI_L2', 'AIFI_L3', 'predicted.celltype.l1', 'predicted.celltype.l2', 'predicted.celltype.l3', 'scDblFinder.class', 'n_genes_by_counts', 'log1p_n_genes_by_counts', 'total_counts', 'log1p_total_counts', 'pct_counts_in_top_20_genes', 'total_counts_hb', 'log1p_total_counts_hb', 'pct_counts_hb', 'High_quality', 'Batch', 'L1_leiden_scVI_3', 'L1_leiden_scVI_4', 'L1_leiden_scVI_5', 'receptor_type', 'receptor_subtype', 'chain_pairing', 'VJ_1_locus', 'VJ_1_v_call', 'VJ_1_j_call', 'VJ_1_junction_aa', 'VJ_1_junction', 'VJ_1_umi_count', 'VDJ_1_locus', 'VDJ_1_v_call', 'V

In [28]:
adata_count = sc.read_h5ad('/home/liyanguo/MyImmuCell/04_Ref_Atlas_scVI/Ref_Atlas_Celltype_L1_L2_scVI.h5ad')

In [29]:
adata_count

AnnData object with n_obs × n_vars = 888260 × 38606
    obs: 'orig.ident', 'nCount_RNA', 'nFeature_RNA', 'SampleID', 'DonorID', 'percent_mito', 'percent_ribo', 'percent_mito_ribo', 'log10GenesPerUMI', 'percent_top50', 'percent_oxphos', 'percent_apop', 'percent_dna_repair', 'percent_ieg', 'percent_hemo', 'S.Score', 'G2M.Score', 'Phase', 'Immune_All_High', 'Immune_All_Low', 'Adult_Human_Blood', 'Adult_Human_Bone_marrow', 'AIFI_L1', 'AIFI_L2', 'AIFI_L3', 'predicted.celltype.l1', 'predicted.celltype.l2', 'predicted.celltype.l3', 'scDblFinder.class', 'n_genes_by_counts', 'log1p_n_genes_by_counts', 'total_counts', 'log1p_total_counts', 'pct_counts_in_top_20_genes', 'total_counts_hb', 'log1p_total_counts_hb', 'pct_counts_hb', 'High_quality', 'Batch', 'L1_leiden_scVI_3', 'L1_leiden_scVI_4', 'L1_leiden_scVI_5', 'receptor_type', 'receptor_subtype', 'chain_pairing', 'VJ_1_locus', 'VJ_1_v_call', 'VJ_1_j_call', 'VJ_1_junction_aa', 'VJ_1_junction', 'VJ_1_umi_count', 'VDJ_1_locus', 'VDJ_1_v_call', 'V

In [30]:
adata_filter = adata_count[adata.obs_names,:]

In [31]:
adata_filter

View of AnnData object with n_obs × n_vars = 717723 × 38606
    obs: 'orig.ident', 'nCount_RNA', 'nFeature_RNA', 'SampleID', 'DonorID', 'percent_mito', 'percent_ribo', 'percent_mito_ribo', 'log10GenesPerUMI', 'percent_top50', 'percent_oxphos', 'percent_apop', 'percent_dna_repair', 'percent_ieg', 'percent_hemo', 'S.Score', 'G2M.Score', 'Phase', 'Immune_All_High', 'Immune_All_Low', 'Adult_Human_Blood', 'Adult_Human_Bone_marrow', 'AIFI_L1', 'AIFI_L2', 'AIFI_L3', 'predicted.celltype.l1', 'predicted.celltype.l2', 'predicted.celltype.l3', 'scDblFinder.class', 'n_genes_by_counts', 'log1p_n_genes_by_counts', 'total_counts', 'log1p_total_counts', 'pct_counts_in_top_20_genes', 'total_counts_hb', 'log1p_total_counts_hb', 'pct_counts_hb', 'High_quality', 'Batch', 'L1_leiden_scVI_3', 'L1_leiden_scVI_4', 'L1_leiden_scVI_5', 'receptor_type', 'receptor_subtype', 'chain_pairing', 'VJ_1_locus', 'VJ_1_v_call', 'VJ_1_j_call', 'VJ_1_junction_aa', 'VJ_1_junction', 'VJ_1_umi_count', 'VDJ_1_locus', 'VDJ_1_v_c

In [32]:
del adata_filter.obs['Celltype_L1_L2_Refine']

In [ ]:
adata_filter.obs['Celltype_L1_L2_Refine']=adata.obs['Celltype_L1_L2_Refine']

In [34]:
adata_filter.X.max()

4753.0

In [ ]:
adata_filter.obs['Celltype_L1_L2_Refine'].value_counts()

In [36]:
adata_filter.obs['Celltype_L1_L2_Refine'].isna().value_counts()

Celltype_L1_L2_Refine
False    717723
Name: count, dtype: int64

In [37]:
sc.settings.figdir = obj_path

In [38]:
adata_filter.obs['receptor_subtype_BCR'].value_counts()

receptor_subtype_BCR
IGH+IGK      13657
IGH+IGL       8846
IGH+IGK/L     1312
IGH           1187
ambiguous       95
Name: count, dtype: int64

In [39]:
adata_filter.obs['receptor_subtype'].value_counts()

receptor_subtype
TRA+TRB    236359
Name: count, dtype: int64

In [ ]:
adata_filter.write(f'{obj_path}Celltype_L1_L2_Refine_R1.h5ad',compression="gzip")

# visualization

In [78]:
adata.write(f'{obj_path}Celltype_L1_L2_Refine_R1.h5ad',compression="gzip")

In [16]:
adata = sc.read_h5ad(f'{obj_path}Celltype_L1_L2_Refine_R1.h5ad')

In [47]:
adata.obs['Celltype_L1_L2_Refine'].value_counts()

Celltype_L1_L2_Refine
NDNs                         240522
CD4+ T cells                 158168
Non-MAIT/NKT CD8+ T cells    114555
NK cells                      79020
Classical monocytes           44552
B cells                       28220
MAIT                          17983
Non-classical monocytes       15652
γδ T                           5109
cDCs                           3096
Proliferative T/NK             2341
Plamsa cells                   1842
Basophils                      1769
pDCs                           1699
LDNs                           1645
Platelets                      1189
HSPC                            261
Mast                             58
iNKT                             42
Name: count, dtype: int64

In [20]:
sc.settings.figdir = obj_path

In [ ]:
sc.pl.umap(adata, color=['Celltype_L1_L2_Refine','receptor_subtype','receptor_subtype_BCR'],
           save='_Celltype_L1_L2_Refine',
           frameon=False,
           legend_fontsize=8,
           size=0.1,legend_loc='on data',
          )

In [49]:
groupby='Celltype_L1_L2_Refine'

In [77]:
xlsx = pd.ExcelWriter(f"{obj_path}{groupby}_freq_table.xlsx")
for group in ['SampleID', 'DonorID',
              'scDblFinder.class','Phase',
              'Immune_All_High', 'Immune_All_Low', 'Adult_Human_Blood', 'Adult_Human_Bone_marrow',
              'AIFI_L1', 'AIFI_L2', 'AIFI_L3', 'predicted.celltype.l1', 'predicted.celltype.l2',
              'predicted.celltype.l3',
              'receptor_subtype', #TCR
              'VJ_1_v_call', 'VJ_1_j_call', 
              'VDJ_1_v_call', 'VDJ_1_j_call',
              'receptor_type_BCR', #BCR
              'VJ_1_v_call_BCR','VJ_1_j_call_BCR', 
              'VDJ_1_v_call_BCR', 'VDJ_1_j_call_BCR']:
    freq_table_multi = adata.obs.groupby([groupby, group]).size()
    pd.DataFrame(freq_table_multi).to_excel(xlsx,sheet_name=group)
xlsx.close()

In [ ]:
#数据簇间信息比较
fig, axs = plt.subplots(3, 2, figsize=(18, 12))#,constrained_layout=True
plt.subplots_adjust(hspace=1,wspace=1)
sc.pl.violin(adata, keys='nCount_RNA', groupby=groupby, rotation=90,size=0,palette="pastel",linewidth=0,ax=axs[0,0], show=False)
sc.pl.violin(adata, keys='nFeature_RNA', groupby=groupby, rotation=90,size=0,palette="pastel",linewidth=0,ax=axs[0,1], show=False)
#sc.pl.violin(adata, keys='log10GenesPerUMI', groupby=groupby, rotation=90,size=0,palette="pastel",linewidth=0,ax=axs[1,0], show=False)
#sc.pl.violin(adata, keys='pct_counts_in_top_20_genes', groupby=groupby, rotation=90,size=0,palette="pastel",linewidth=0,ax=axs[1,1], show=False)

sc.pl.violin(adata, keys='percent_apop', groupby=groupby, rotation=90,size=0,palette="pastel",linewidth=0,ax=axs[1,0], show=False)
sc.pl.violin(adata, keys='percent_ribo', groupby=groupby, rotation=90,size=0,palette="pastel",linewidth=0,ax=axs[1,1], show=False)
sc.pl.violin(adata, keys='percent_ieg', groupby=groupby, rotation=90,size=0,palette="pastel",linewidth=0,ax=axs[2,0], show=False)
sc.pl.violin(adata, keys='percent_oxphos', groupby=groupby, rotation=90,size=0,palette="pastel",linewidth=0,ax=axs[2,1], show=False)
#sc.pl.violin(adata, keys='percent_hemo', groupby=groupby, rotation=90,size=0,palette="pastel",linewidth=0,ax=axs[4,0], show=False)
#sc.pl.violin(adata, keys='G2M.Score', groupby=groupby, rotation=90,size=0,palette="pastel",linewidth=0,ax=axs[4,1], show=False)

fig.tight_layout()
plt.savefig(f'{obj_path}{groupby}_scVI_L2_metadata.pdf')

In [76]:
xlsx = pd.ExcelWriter(f"{obj_path}{groupby}_QC_info_.xlsx")
for group in ['nCount_RNA', 'nFeature_RNA','percent_mito', 'percent_ribo', 'percent_mito_ribo',
              'log10GenesPerUMI', 'percent_top50', 'percent_oxphos', 'percent_apop',
              'percent_dna_repair', 'percent_ieg', 'percent_hemo',groupby]:
    data_table_multi = adata.obs[groupby].describe()
    pd.DataFrame(data_table_multi).to_excel(xlsx,sheet_name=group)
xlsx.close()

## Umap of different sample type

In [52]:
adata.obs['type'] = adata.obs.SampleID.str.split('_',expand=True).loc[:,1]

In [53]:
split = adata.obs['type'].unique().tolist()
split

['Mix', 'WB', 'PBMC']

In [ ]:
fig, ax = plt.subplots(nrows=1, ncols=3, squeeze=False, figsize=(7 * 3, 5))

tmp = adata[adata.obs['type'] == 'PBMC', :]
sc.pl.umap(tmp, color='Celltype_L1_L2_Refine', ax=ax[0, 0],
       legend_loc=None,frameon=False,show=False)
ax[0, 0].set_title(f"PBMC")

tmp = adata[adata.obs['type'] == 'WB', :]
sc.pl.umap(tmp, color='Celltype_L1_L2_Refine', ax=ax[0, 1],
       legend_loc=None,frameon=False,show=False)
ax[0, 1].set_title(f"WB")

tmp = adata[adata.obs['type'] == 'Mix', :]
sc.pl.umap(tmp, color='Celltype_L1_L2_Refine', ax=ax[0, 2],
       legend_fontsize=6,frameon=False,show=False)
ax[0, 2].set_title(f"Mix")

In [ ]:
fig.subplots_adjust()
fig.savefig(f'{obj_path}split_sample_L2_umap.pdf')

In [59]:
plt

<module 'matplotlib.pyplot' from '/home/liyanguo/anaconda3/envs/R45/lib/python3.13/site-packages/matplotlib/pyplot.py'>

In [ ]:
marker_gene_dict={
                  'LDNs':['CEACAM8','CEBPE',
                              'PADI4','CD177',
                              'MPO','ELANE',
                              'LTF',
                              'MMP8','MMP9','IL17RA','CD38'
                                ],
                  
                  'NDNs':['CEBPB','FCGR3B','MME','SELL','CSF3R','CXCR2','CXCL8','CXCR4','NAMPT'],
                  'Subset':['TNFSF10','ISG15',
                        'IRF1','GBP2',
                        'FOS','JUN','DUSP1',
                        'RGS3'],
    
                  'Immune checkpoints':['VSIR','ARG1','CD47','SIRPA',
                                        'CD24',
                                       ],

                  'NADPH':['CYBB'],

                  'Proliferative':['MKI67'],

                  'Complement':['CD55','C5AR1','CR1'],
    'Other':['TNFRSF14','CXCL1'],
                 }

In [ ]:
ax = sc.pl.dotplot(
    adata, marker_gene_dict, groupby='Celltype_L1_L2_Refine', swap_axes=False,
    dendrogram=False,cmap='Spectral_r',
    colorbar_title="Column scaled\nexpression",
    standard_scale='var'
)